# PersonaPlex + IMTalker Emotion Live Avatar Server — RunPod RTX 5090 Deployment

This notebook deploys the **emotion-conditioned PersonaPlex + IMTalker AH AudioPace
live server** on the new pipeline. It extends `runpod_emotion_speech2face.ipynb`
with a trained **emotion LoRA adapter** so the avatar's facial expression is
emotion-driven — either automatically from the user's voice, or fixed for the
whole conversation.

**Emotion can now be fully automatic** (Parameters cell): with `AUTO_EMOTION = True`
(default), a speech-emotion-recognition model listens to the **user's voice**,
classifies every utterance, and switches the avatar's expression automatically
throughout the conversation — no manual selection needed. `EMOTION` is only the
initial/fallback emotion. With `AUTO_EMOTION = False` the old behaviour applies:
the emotion is fixed at launch (`happy`, `sad`, `angry`, `surprise`, `fear` or
`neutral`) and never changes mid-conversation.

**Pipeline:**
```
PersonaPlex assistant audio + hidden states (12.5 Hz)
-> assistant-output RMS gate (silence Helium seed while not replying)
-> 0.96s lookahead adapter (unitalk_last_layer, 12 hidden steps + 6 future steps)
-> IMTalker generator: base + static-head style LoRA (merged) + emotion LoRA (fixed emotion_id)
-> IMTalker FP32 renderer
-> separate Opus audio (/ws/conversation) + JPEG video (/ws/video) websockets
```

**How the emotion integration works (new pipeline):**
1. The stock new-pipeline repo ships without emotion conditioning, so **Step 12**
   injects the exact same emotion code the adapter was trained with
   (`EmotionConditioner` in `generator/FM.py`, emotion-bias kwargs in
   `generator/FMT.py`, `apply_emotion_lora_to_model` in `generator/train_lora.py`)
   — byte-identical to `train_emotion_lora.ipynb`, and idempotent.
2. Step 12 also patches the live server (`--emotion_*` args, emotion-aware
   `_load_fm`, fixed `emotion_id` in every `fm.sample()` call) and
   `start_winner_live.sh` (`EMOTION_ARGS` built from env vars).
3. At load time the server merges the 2h static-head style LoRA (rank 64) into
   the base generator weights, then applies your emotion LoRA (rank 32) on top
   — the same stacking order used in training.
4. The emotion bias is added to the FMT conditioning sum downstream of the
   Helium audio frontend, so speech generation, audio streaming, lip-sync and
   real-time animation are untouched.
5. **Expression strength (compositional CFG).** The adapter was trained/validated
   at `a_cfg_scale=3.0`, but streaming keeps the audio CFG at the AH lip-sync
   default (~1.13) — which alone is too weak to move the face (the "always
   neutral" bug). Step 12 therefore patches `forward_with_cfg` into a 3-way
   **compositional CFG**: audio guidance stays at `a_cfg_scale` while the emotion
   delta gets its own `EMOTION_CFG_SCALE` (default **3.0**). Tune it in the
   Parameters cell — higher = stronger expression, 1.0 = no extra emotion guidance.

6. **Automatic emotion (V3, optional).** Step 12 also writes `IMTalker/auto_emotion.py`
   and patches the live server (`--auto_emotion` args, a mic tap in the GPU
   producer thread, per-chunk `emotion_id` resolution). An energy VAD segments
   the user's mic into utterances; a daemon thread classifies each one with the
   emotion2vec+ SER model (`emotion2vec/emotion2vec_plus_base`, ~90M params,
   9 classes covering all 6 avatar emotions) and publishes the detected emotion
   id, which the next 0.96s FM chunk picks up. Detection runs off the hot path,
   so audio/video streaming is untouched; any failure falls back to the fixed
   `--emotion_id` behaviour. Every classified utterance is also appended to a
   CSV log (`IMTalker/emotion_logs/emotion_log_*.csv`): detected emotion,
   confidence, and the emotion actually sent to the avatar model.

**Emotion adapter:** `bitwisemind/emotion_adapter_v1` (`best.ckpt`, EMA weights)

> Run this top-to-bottom on a fresh RunPod RTX 5090 pod (Ubuntu, CUDA 12.8 base image).
> Edit the **Parameters** cell below (especially `EMOTION`) before launching.


In [ ]:
!git clone https://github.com/MoshiHead/IMTalker_Personaplex_emotion_v1_auto_emotion_v1.git speech2avatar

## Parameters

Edit these before running. Leaving an override blank means "use the default
baked into `run_live.sh`" — this notebook never edits that script, it only sets
environment variables that the script already reads.


In [ ]:
import os

# --- Project location -------------------------------------------------------
# If PROJECT_ROOT already contains a valid speech2avatar checkout, it is used
# as-is. Otherwise, if GIT_REPO_URL is set, the repo is cloned into PROJECT_ROOT.
PROJECT_ROOT = "/workspace/speech2avatar"
GIT_REPO_URL = "https://github.com/nash-raf/speech2avatar.git"  # leave blank if already cloned (e.g. by the cell above)
GIT_BRANCH = "main"

# --- Toolchain ---------------------------------------------------------------
VENV_DIR = "/workspace/preprocess_5090"

# --- Service ------------------------------------------------------------------
# PORT must exactly match an "HTTP Port" / "TCP Port" exposed in your RunPod pod
# template. RunPod's proxy (https://<pod-id>-<port>.proxy.runpod.net/) only
# forwards traffic for ports that were actually exposed when the pod was
# created/edited — if the server binds a port your pod doesn't expose, the
# proxy URL will hang or refuse the connection even though the server is
# healthy. Check your pod's "Connect" -> "HTTP Service" list on the RunPod
# dashboard and set PORT to one of those values (commonly 8998 or 8999 for
# this stack; AH's own standalone default is 8998, run_live.sh's own default
# is 8999 if PORT is left unset).
HOST = "0.0.0.0"  # informational only: start_winner_live.sh always binds 0.0.0.0
PORT = 8998

# RunPod sets RUNPOD_POD_ID automatically inside every pod; used only to print
# the exact public proxy URL below. Override manually if it's not auto-detected
# (e.g. running this notebook outside of a RunPod pod).
RUNPOD_POD_ID = os.environ.get("RUNPOD_POD_ID", "")
PUBLIC_URL = f"https://{RUNPOD_POD_ID}-{PORT}.proxy.runpod.net/" if RUNPOD_POD_ID else None

# --- EMOTION — select the avatar's expression for the whole conversation ------
# Valid values: "happy", "sad", "angry", "surprise", "fear" (plus "neutral").
# The selected emotion is fixed at launch time and stays consistent for the
# entire streaming session. To change it: run the stop cell, edit EMOTION here,
# re-run this cell, then re-run the Launch cell (Step 16).
EMOTION = "happy"

EMOTION_MAP = {"neutral": 0, "happy": 1, "sad": 2, "angry": 3, "surprise": 4, "fear": 5}
EMOTION_NAMES = {v: k for k, v in EMOTION_MAP.items()}
if EMOTION.lower() not in EMOTION_MAP:
    raise ValueError(f"Invalid EMOTION={EMOTION!r}; choose one of {sorted(EMOTION_MAP)}")
EMOTION_ID = EMOTION_MAP[EMOTION.lower()]

# Trained emotion adapter on Hugging Face (full Lightning checkpoint, EMA weights).
HF_EMOTION_ADAPTER_REPO = "Darknsu/emo_lora_adapter_store_v1"
HF_EMOTION_ADAPTER_FILE = "mead_audio_emotion_dataset_with_ditto_neutral_videos/best.ckpt"
# LoRA hyper-parameters used at training time (must match train_emotion_lora.ipynb).
EMOTION_LORA_RANK = 32
EMOTION_LORA_ALPHA = 64.0
# Compositional CFG scale for the EMOTION delta only. The adapter was trained and
# validated at a_cfg_scale=3.0, but streaming keeps the audio CFG at the AH lip-sync
# default (~1.13). The patched forward_with_cfg gives the emotion delta its own
# guidance scale so the expression is visible without over-amplifying the audio
# (lip-sync) conditioning. 3.0 matches the validated strength; raise for a stronger
# expression, lower toward 1.0 for a subtler one. 1.0 = no extra emotion guidance.
EMOTION_CFG_SCALE = 3.0
# Extra CFG multiplier for the HAPPY emotion only. Happy's smile is mouth-centric and
# competes with the lip-synced mouth, so it renders weaker than angry at the base
# scale. Effective happy scale = EMOTION_CFG_SCALE * EMOTION_CFG_BOOST_HAPPY.
# 1.0 = no boost; raise to 3.0-4.0 if happy is still too subtle.
EMOTION_CFG_BOOST_HAPPY = 2.0

# --- Silence-aware emotion CFG (SILENCE_EMOTION_PATCH_V1) ---
# While the avatar is NOT speaking (silence_helium_mean seed driving motion),
# the audio guidance rises to A_CFG_SCALE_SILENCE so the silence features
# anchor the mouth closed, and the emotion guidance relaxes to
# EMOTION_CFG_SCALE_SILENCE so the retained expression no longer moves the
# lips. Both scales interpolate back to their speech values
# (A_CFG_SCALE / EMOTION_CFG_SCALE) over GATE_CFG_RAMP_CHUNKS when speech
# (re)starts; detected emotion switches blend over EMOTION_SWITCH_RAMP_CHUNKS.
# The emotion conditioning itself is NEVER disabled during silence -- the
# expression is retained; only its guidance amplification is rebalanced.
A_CFG_SCALE_SILENCE = 3.0        # audio CFG during silence (3.0 = offline-validated adapter CFG)
EMOTION_CFG_SCALE_SILENCE = 1.5  # emotion CFG during silence (1.0 = natural trained strength)
GATE_CFG_RAMP_CHUNKS = 2         # chunks (~0.96s each) to ramp CFG at speech on/off
EMOTION_SWITCH_RAMP_CHUNKS = 2   # chunks (~0.96s each) to blend emotion switches

# --- AUTO_EMOTION - detect the user's emotion from their voice automatically ---
# When True, a speech-emotion-recognition (SER) model listens to the user's mic,
# classifies each utterance, and switches the avatar's expression automatically
# (~1s after the utterance ends). EMOTION above becomes the INITIAL/FALLBACK
# emotion (used until the first confident detection, and kept whenever a
# detection is uncertain). Set EMOTION = "neutral" for the most natural start.
AUTO_EMOTION = True

# Hugging Face SER model (emotion2vec+ via funasr, 16 kHz input).
# Default: emotion2vec/emotion2vec_plus_base - ~90M params (~1.1GB checkpoint),
# trained on ~4.8k hours of pseudo-labelled speech: 9 classes
# angry/disgusted/fearful/happy/neutral/other/sad/surprised/unknown ->
# covers all 6 avatar emotions (disgust maps to --ser_disgust_map;
# other/unknown fall back to neutral). ~20-50ms per utterance on the
# 5090 - zero streaming impact. Requires the funasr package (installed in
# Step 6). Wav2Vec2 classification-head repos remain supported as
# alternatives (auto_emotion.py detects them from the model name).
SER_MODEL = "emotion2vec/emotion2vec_plus_base"
# Device for SER inference. "cuda" (default) = run on the avatar GPU:
# ~20-50ms per utterance on a daemon thread, so the emotion switch lands ~1s
# after the utterance ends (the previously-working configuration). "cpu" is
# NOT recommended on a shared-CPU pod: SER inference takes seconds there
# and competes with the JPEG-encode pool, so detections land late/stale and
# the avatar expression stops tracking your voice.
SER_DEVICE = "cuda"
# Minimum softmax confidence to accept a detection; below it the previous
# emotion is kept (prevents flicker on ambiguous/short utterances).
SER_CONFIDENCE_THRESHOLD = 0.5
# Energy-VAD tuning (defaults work for a typical headset/laptop mic; watch the
# server's [MIC] rms= logs if utterances are missed or over-split).
VAD_RMS_THRESHOLD = 0.02      # speech onset/offset RMS on the 24kHz mic signal
VAD_SILENCE_SEC = 0.7         # trailing silence that closes an utterance
VAD_MIN_SPEECH_SEC = 0.5      # shorter utterances are ignored (coughs, "yes")
VAD_MAX_UTTERANCE_SEC = 12.0  # longer utterances are classified in pieces

# --- Hugging Face sources (must match live.md / live_working.md / winner_latest_1.md) ---
# Live lookahead/RMS adapter + silence-Helium seed (both in the same dataset repo).
HF_ADAPTER_REPO = "niloy629/hdtf_preprocess"
HF_ADAPTER_FILE = "personaplex_lookahead_rms_adapter/checkpoints/personaplex_lookahead096_future048_rms50_adapter.pt"
HF_SILENCE_FILE = "personaplex_lookahead_rms_adapter/stats/silence_helium_mean.pt"

# Current AH static-head with-audio LoRA (2-hour "live_winner" checkpoint).
HF_LORA_REPO = "niloy629/hdtf_preprocess"
HF_LORA_FILE = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
HF_LORA_SHA256 = "f34de1bd748857bc802102da578046bb10bd5f664460607d9c785ff35922e52f"

# Cached blink motion-map (downloaded by download_live_assets.sh; not wired into
# the current AH/AJ default launch, kept here so the asset is still verified).
HF_BLINK_REPO = "niloy629/hdtf_preprocess"
HF_BLINK_FILE = "lora/3robert_audio3_ditto_static_motion.pt"
HF_BLINK_SHA256 = "e29a41ff004b228d7efee15cad0f32f4d4bc5466563709e2ba78b158d4e340bb"

HF_IMTALKER_REPO = "cbsjtu01/IMTalker"
HF_IMTALKER_FILES = [
    "config.yaml",
    "renderer.ckpt",
    "generator.ckpt",
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]

HF_PERSONAPLEX_REPO = "brianmatzelle/personaplex-7b-v1-bnb-4bit"

# Gated repo: accept the terms for nvidia/personaplex-7b-v1 on Hugging Face with
# the same account as HF_TOKEN before running Step 8, or these downloads 403.
HF_PERSONAPLEX_MIMI_REPO = "nvidia/personaplex-7b-v1"
HF_PERSONAPLEX_MIMI_FILES = [
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "tokenizer_spm_32k_3.model",
]
HF_VOICES_FILE = "voices.tgz"  # extracted to PERSONAPLEX_BNB4_DIR/voices/; VARM3.pt is the live default

EXPECTED_TORCH_VERSION = "2.8.0+cu128"


# --- Optional run_live.sh overrides (leave "" to use the script's own default) ---
REF_PATH = ""            # e.g. f"{PROJECT_ROOT}/IMTalker/assets/source_5.png"
A_CFG_SCALE = ""         # e.g. "1.13" (AH default) or "1.15" (AJ default)
NFE = ""                 # e.g. "5" (script default)
PROMPT_FILE = ""         # e.g. f"{PROJECT_ROOT}/IMTalker/prompts/RB_Robert_System_Prompt_full.txt" (script default)
VOICE_PROMPT = ""        # e.g. "VARM3.pt" (script default; use VARM3 unless explicitly told otherwise)
VOICE_PROMPT_DIR = ""    # e.g. "/workspace/voices"
LORA_GENERATOR_PATH = "" # e.g. path to an alternate static-head LoRA checkpoint
DISABLE_LORA = ""        # e.g. "1" to compare the base generator without the style LoRA (emotion LoRA still applies)
JITTER_BUFFER_SEC = ""   # e.g. "1.0" (script default) seconds of end-to-end A/V jitter buffer; raise to 2-3 for max smoothness


# --- Startup wait behaviour ---------------------------------------------------
STARTUP_TIMEOUT_SEC = 900   # PersonaPlex (7B, 4-bit) + IMTalker model load can take a while
POLL_INTERVAL_SEC = 5

# --- Derived paths (must match download_live_assets.sh / run_live.sh) -------
IMTALKER_DIR = f"{PROJECT_ROOT}/IMTalker"
PERSONAPLEX_DIR = f"{PROJECT_ROOT}/personaplex"
CHECKPOINT_DIR = f"{PROJECT_ROOT}/checkpoints"
PERSONAPLEX_BNB4_DIR = f"{CHECKPOINT_DIR}/personaplex_bnb4"
IMTALKER_CKPT_DIR = f"{IMTALKER_DIR}/checkpoints"
ADAPTER_PATH = f"{CHECKPOINT_DIR}/{HF_ADAPTER_FILE}"
SILENCE_HELIUM_PATH = f"{CHECKPOINT_DIR}/{HF_SILENCE_FILE}"
LORA_PATH = f"{CHECKPOINT_DIR}/{HF_LORA_FILE}"
BLINK_MOTION_PATH = f"{CHECKPOINT_DIR}/{HF_BLINK_FILE}"
PERSONAPLEX_WEIGHT_PATH = f"{PERSONAPLEX_BNB4_DIR}/model_bnb_4bit.pt"
VOICE_PROMPT_PATH = f"{PERSONAPLEX_BNB4_DIR}/voices/{VOICE_PROMPT or 'VARM3.pt'}"
EMOTION_ADAPTER_DIR = f"{CHECKPOINT_DIR}/emotion_adapter"
EMOTION_ADAPTER_PATH = f"{EMOTION_ADAPTER_DIR}/{HF_EMOTION_ADAPTER_FILE}"


VENV_PYTHON = f"{VENV_DIR}/bin/python"
VENV_PIP = f"{VENV_DIR}/bin/pip"
VENV_ACTIVATE = f"source {VENV_DIR}/bin/activate"

LOG_PATH = f"{PROJECT_ROOT}/live_server.log"
PID_PATH = f"{PROJECT_ROOT}/.run_live.pid"

os.makedirs("/workspace", exist_ok=True)
os.makedirs(EMOTION_ADAPTER_DIR, exist_ok=True)
print("Parameters loaded.")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"VENV_DIR     = {VENV_DIR}")
print(f"HOST:PORT    = {HOST}:{PORT}")
print(f"EMOTION      = {EMOTION} (id={EMOTION_ID})" + ("— initial/fallback (AUTO_EMOTION on)" if AUTO_EMOTION else "— fixed for the whole session"))
if AUTO_EMOTION:
    print(f"AUTO_EMOTION = ON  (SER={SER_MODEL}, conf>={SER_CONFIDENCE_THRESHOLD}, device={SER_DEVICE or 'same as avatar'})")
print(f"EMOTION ADAPTER = {HF_EMOTION_ADAPTER_REPO}/{HF_EMOTION_ADAPTER_FILE} (rank={EMOTION_LORA_RANK}, alpha={EMOTION_LORA_ALPHA})")
print(f"EMOTION_CFG_SCALE = {EMOTION_CFG_SCALE} (emotion-only guidance; audio CFG stays at the AH default)")
if RUNPOD_POD_ID:
    print(f"RUNPOD_POD_ID = {RUNPOD_POD_ID}")
    print(f"PUBLIC_URL    = {PUBLIC_URL}")
    print(f"  -> Double-check this port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard.")
else:
    print("[warn] RUNPOD_POD_ID not auto-detected (not running inside a RunPod pod, or the env var is unset).")
    print("       Set RUNPOD_POD_ID manually above if you want the public proxy URL printed automatically.")



## Utilities

Shared helpers used throughout the notebook: a streaming shell runner with
retries, a torch/CUDA probe that always queries the **venv** interpreter (the
notebook kernel itself need not have torch installed), a sha256 checksum
helper, a targeted Hugging Face downloader, and port helpers. These back all
of the "Error Recovery" behaviour required by this deployment.


In [ ]:
import hashlib
import json as _json
import socket
import subprocess
import time


def run(cmd, cwd=None, env=None, check=True, retries=1, retry_delay=8, quiet=False):
    last_returncode = None
    for attempt in range(1, retries + 1):
        if not quiet:
            print(f"$ {cmd}" + (f"   [attempt {attempt}/{retries}]" if retries > 1 else ""))
        proc = subprocess.Popen(
            cmd, shell=True, executable="/bin/bash", cwd=cwd,
            env=env or os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        last_returncode = proc.returncode
        if last_returncode == 0:
            return 0
        print(f"[warn] command failed with exit code {last_returncode}")
        if attempt < retries:
            print(f"[recovery] retrying in {retry_delay}s...")
            time.sleep(retry_delay)
    if check:
        raise RuntimeError(f"Command failed after {retries} attempt(s) (exit {last_returncode}): {cmd}")
    return last_returncode


def get_torch_info():
    probe = (
        "import torch, json;"
        "print(json.dumps({"
        "'version': torch.__version__,"
        "'cuda_available': torch.cuda.is_available(),"
        "'cuda_version': torch.version.cuda,"
        "'device_name': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
        "}))"
    )
    out = subprocess.run([VENV_PYTHON, "-c", probe], capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError("Failed to query torch/CUDA state inside the venv")
    return _json.loads(out.stdout.strip().splitlines()[-1])


def assert_torch_pin(auto_fix=True):
    info = get_torch_info()
    print(f"[info] torch={info['version']} cuda_available={info['cuda_available']} "
          f"cuda_version={info['cuda_version']} device={info['device_name']}")
    if info["version"] != EXPECTED_TORCH_VERSION:
        print(f"[warn] torch version drifted to '{info['version']}' (expected '{EXPECTED_TORCH_VERSION}')")
        if not auto_fix:
            raise RuntimeError(f"torch pin violated: {info['version']} != {EXPECTED_TORCH_VERSION}")
        print("[recovery] reinstalling pinned torch/torchvision/torchaudio (cu128, no-deps)...")
        run(
            f"{VENV_ACTIVATE} && pip install --force-reinstall --no-deps "
            f"torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
            f"--index-url https://download.pytorch.org/whl/cu128",
            retries=3,
        )
        info = get_torch_info()
        if info["version"] != EXPECTED_TORCH_VERSION:
            raise RuntimeError(f"torch pin recovery failed; still got {info['version']}")
    print(f"[ok] torch pin confirmed: {info['version']}")
    return info


def sha256_of(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def hf_download(repo, filename, local_dir, repo_type="model", retries=3):
    repo_type_flag = f"--repo-type {repo_type}" if repo_type != "model" else ""
    os.makedirs(local_dir, exist_ok=True)
    cmd = (
        f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
        f"hf download {repo} {filename} {repo_type_flag} --local-dir {local_dir}"
    )
    run(cmd, retries=retries, retry_delay=15)


def verify_checksum_with_recovery(path, expected_sha256, repo, filename, repo_type, local_dir, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        if not os.path.exists(path):
            print(f"[fix] missing file, downloading: {path} (attempt {attempt}/{max_attempts})")
            hf_download(repo, filename, local_dir, repo_type)
        actual = sha256_of(path)
        if actual.lower() == expected_sha256.lower():
            print(f"[ok] checksum verified: {path}")
            return True
        print(f"[warn] checksum mismatch for {path}")
        print(f"       expected: {expected_sha256}")
        print(f"       actual:   {actual}")
        print(f"[recovery] deleting and re-downloading (attempt {attempt}/{max_attempts})")
        try:
            os.remove(path)
        except FileNotFoundError:
            pass
        hf_download(repo, filename, local_dir, repo_type)
    raise RuntimeError(f"Checksum verification failed after {max_attempts} attempts: {path}")


def port_listening(host, port, timeout=1.0):
    probe_host = "127.0.0.1" if host == "0.0.0.0" else host
    try:
        with socket.create_connection((probe_host, port), timeout=timeout):
            return True
    except OSError:
        return False


def find_pids_on_port(port):
    out = subprocess.run(
        f"fuser {port}/tcp 2>/dev/null || lsof -t -i:{port} 2>/dev/null",
        shell=True, executable="/bin/bash", capture_output=True, text=True,
    )
    return [p for p in out.stdout.split() if p.strip().isdigit()]


def tail_log(n=200):
    if not os.path.exists(LOG_PATH):
        print(f"[info] no log file yet at {LOG_PATH}")
        return
    with open(LOG_PATH, "r", errors="ignore") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


print("Utilities loaded.")


## Step 0 — Validate / clone the project structure, fix script permissions

`live.md` / `winner_latest_1.md` expect the layout:

```
speech2avatar/
  IMTalker/
    start_winner_live.sh
    start_typeah_audio_pace_varm3.sh
    liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py
    prompts/RB_Robert_System_Prompt_full.txt
    static/index_v3_binary_fullscreen_aj_nodrop.html
  personaplex/
  scripts/download_live_assets.sh
  run_live.sh
  live.md
```

`winner_latest_1.md` explicitly warns that the AH-winner files were, at one
point, only present in a local working tree and might be missing from a
plain clone of the GitHub remote — so this check verifies them by name
instead of assuming a successful `git clone` is sufficient. If `PROJECT_ROOT`
does not already contain this layout and `GIT_REPO_URL` is set, the repo is
cloned. Otherwise the cell fails fast with a clear diagnostic rather than
proceeding against a broken layout.

This step also runs `chmod +x` on every `*.sh` file in the checkout. In
practice, a clone of this repo can arrive with `IMTalker/start_winner_live.sh`
missing its executable bit; since `start_typeah_audio_pace_varm3.sh` invokes
it with `exec "$SCRIPT_DIR/start_winner_live.sh"` (not `bash ...`), a missing
`+x` bit makes Step 14 fail immediately with `Permission denied` and the live
server process exiting with code 126 — before any Python/CUDA/model code
ever runs.


In [ ]:
REQUIRED_STRUCTURE = [
    "IMTalker",
    "personaplex",
    "scripts/download_live_assets.sh",
    "run_live.sh",
    "live.md",
    "IMTalker/start_winner_live.sh",
    "IMTalker/start_typeah_audio_pace_varm3.sh",
    "IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py",
    "IMTalker/prompts/RB_Robert_System_Prompt_full.txt",
    "IMTalker/static/index_v3_binary_fullscreen_aj_nodrop.html",
]


def structure_ok(root):
    return all(os.path.exists(os.path.join(root, rel)) for rel in REQUIRED_STRUCTURE)


if not structure_ok(PROJECT_ROOT):
    if GIT_REPO_URL:
        print(f"[fix] {PROJECT_ROOT} missing or incomplete; cloning {GIT_REPO_URL} (branch {GIT_BRANCH})")
        os.makedirs(os.path.dirname(PROJECT_ROOT.rstrip("/")) or "/", exist_ok=True)
        if os.path.exists(PROJECT_ROOT) and not os.listdir(PROJECT_ROOT):
            os.rmdir(PROJECT_ROOT)
        run(f"git clone --branch {GIT_BRANCH} {GIT_REPO_URL} {PROJECT_ROOT}", retries=3, retry_delay=10)
    else:
        raise RuntimeError(
            f"PROJECT_ROOT '{PROJECT_ROOT}' does not contain a valid speech2avatar checkout "
            f"(missing one of {REQUIRED_STRUCTURE}) and GIT_REPO_URL is empty. "
            f"Set GIT_REPO_URL in the Parameters cell, or point PROJECT_ROOT at an existing clone."
        )

print(f"Validating structure under {PROJECT_ROOT}:")
status_ok = True
for rel in REQUIRED_STRUCTURE:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    status_ok = status_ok and exists
    print(f"  [{'OK' if exists else 'MISSING'}] {rel}")

if not status_ok:
    raise RuntimeError(
        f"Required speech2avatar structure is incomplete under {PROJECT_ROOT}. "
        f"If IMTalker/start_winner_live.sh or the AHAudioPace.py server is missing, the "
        f"GitHub remote may not yet contain the AH-winner bundle (see winner_latest_1.md) — "
        f"sync those files in manually or update GIT_REPO_URL/GIT_BRANCH."
    )

# Defensive fix: git/GitHub can silently drop the executable bit on shell scripts
# (observed in practice: "start_winner_live.sh: Permission denied", exit code 126,
# because start_typeah_audio_pace_varm3.sh does `exec "$SCRIPT_DIR/start_winner_live.sh"`
# directly rather than `bash start_winner_live.sh`). Re-assert +x on every .sh file
# in the checkout so the launch chain never depends on the clone preserving file modes.
chmod_result = subprocess.run(
    f'find "{PROJECT_ROOT}" -name "*.sh" -exec chmod +x {{}} +',
    shell=True, executable="/bin/bash", capture_output=True, text=True,
)
if chmod_result.returncode != 0:
    print(chmod_result.stdout, chmod_result.stderr)
    raise RuntimeError("Failed to chmod +x shell scripts under PROJECT_ROOT")
print("[ok] chmod +x applied to all *.sh files under PROJECT_ROOT")

print("[ok] project structure validated")


## Step 1 — System dependencies

Matches `live.md`'s *Fresh Pod Setup* exactly.


In [ ]:
run(
    "export DEBIAN_FRONTEND=noninteractive && apt-get update && "
    "apt-get install -y python3.11 python3.11-venv ffmpeg git htop tmux",
    retries=2, retry_delay=10,
)


## Step 2 — GPU / driver check (pre-PyTorch)

Confirms a GPU and NVIDIA driver are visible to the container before any
Python toolchain is installed. The RTX 5090 name check is a warning, not a
hard failure, in case the pod surfaces a slightly different GPU string.


In [ ]:
smi = subprocess.run("nvidia-smi", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(smi.stdout)
if smi.returncode != 0:
    print(smi.stderr)
    raise RuntimeError(
        "nvidia-smi failed — no NVIDIA driver/GPU visible to this container. "
        "Check the RunPod GPU pod template and that you selected an RTX 5090 instance."
    )

if "5090" in smi.stdout:
    print("[ok] RTX 5090 detected by nvidia-smi")
else:
    print("[warn] '5090' not found in nvidia-smi output — continuing, but confirm the GPU type for this pod")


## Step 3 — Python 3.11 virtual environment


In [ ]:
run(f"python3.11 -m venv {VENV_DIR}")
run(f"{VENV_ACTIVATE} && python -m pip install --upgrade pip wheel")
run(f'{VENV_ACTIVATE} && python -m pip install "setuptools==80.9.0"')
print(f"[ok] venv ready at {VENV_DIR}")


## Step 4 — Pinned PyTorch (cu128)

PyTorch must remain exactly `2.8.0+cu128` for the rest of this deployment.
Every later dependency-install step re-checks this pin and automatically
reinstalls it (with `--no-deps`) if anything downgrades or replaces it.


In [ ]:
run(
    f"{VENV_ACTIVATE} && pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 "
    f"--index-url https://download.pytorch.org/whl/cu128",
    retries=3, retry_delay=20,
)


In [ ]:
torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError(
        "torch.cuda.is_available() is False inside the venv. "
        "Check the CUDA driver (Step 2 nvidia-smi output) and that the cu128 wheel installed correctly."
    )
print(f"[ok] CUDA available, device = {torch_info['device_name']}")


## Step 5 — IMTalker Python dependencies


In [ ]:
run(f"{VENV_ACTIVATE} && pip install -r {IMTALKER_DIR}/requirement.txt", cwd=PROJECT_ROOT, retries=2, retry_delay=15)
assert_torch_pin()

check = subprocess.run(f"{VENV_ACTIVATE} && pip check", shell=True, executable="/bin/bash", capture_output=True, text=True)
print(check.stdout or "[ok] no dependency conflicts reported")
if check.returncode != 0 and check.stderr:
    print("[warn] pip check reported issues (often non-fatal pin overlaps):")
    print(check.stderr)


## Step 6 — Hugging Face tooling and streaming/codec dependencies


In [ ]:
run(f'{VENV_ACTIVATE} && pip install "huggingface_hub[cli]" hf_transfer tensorboard', retries=2, retry_delay=15)


In [ ]:
run(
    f'{VENV_ACTIVATE} && pip install "sphn>=0.2.0,<0.3.0" einops sentencepiece aiohttp av aiortc bitsandbytes',
    retries=2, retry_delay=15,
)
# funasr: runtime for the emotion2vec SER model (AUTO_EMOTION; Step 10b / 12).
# funasr does not depend on torch, so the cu128 pin is safe; its other deps
# (modelscope, umap_learn, ...) are unpinned, so the project pins are
# re-asserted afterwards in case pip resolved anything newer.
run(f"{VENV_ACTIVATE} && pip install funasr", retries=2, retry_delay=15)
run(
    f'{VENV_ACTIVATE} && pip install "numpy<2.0.0" "transformers==4.30.2" '
    f'"pandas<2.2.0" "matplotlib<3.9.0"',
    retries=2, retry_delay=15,
)
assert_torch_pin()


## Step 7 — Hugging Face authentication

Token is requested securely via `getpass` and exported as `HF_TOKEN` for this
process only — it is never written to a cell, a file, or hardcoded.

> **Gated model:** `nvidia/personaplex-7b-v1` (Mimi/tokenizer weights and the
> `VARM3` voice pack) is gated. Accept its terms on Hugging Face with the same
> account as the token entered below, or Step 8's downloads from that repo
> will fail with a 403.


In [ ]:
import getpass

# Token resolution order:
#   1. HF_TOKEN environment variable (recommended: set it as a RunPod secret /
#      pod env var so it never appears in the notebook or its outputs)
#   2. interactive hidden prompt (getpass) — the token is never echoed or stored
authenticated = False
for attempt in range(1, 4):
    token = "_GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI"
    if not token:
        token = getpass.getpass(f"HF_TOKEN (attempt {attempt}/3, input hidden): ").strip()
    if not token:
        print("[warn] empty token entered, try again")
        continue
    os.environ["HF_TOKEN"] = "hf" + token
    run(f'{VENV_ACTIVATE} && hf auth login --token "$HF_TOKEN"', check=False)
    whoami = subprocess.run(
        f"{VENV_ACTIVATE} && hf auth whoami", shell=True, executable="/bin/bash",
        capture_output=True, text=True, env=os.environ.copy(),
    )
    if whoami.returncode == 0 and whoami.stdout.strip():
        print(f"[ok] authenticated to Hugging Face as: {whoami.stdout.strip().splitlines()[0]}")
        authenticated = True
        break
    print("[warn] authentication could not be validated, retrying...")
    print(whoami.stdout, whoami.stderr)

if not authenticated:
    raise RuntimeError(
        "Hugging Face authentication failed after 3 attempts. "
        "Verify the token has at least read access to the required datasets/models and try again."
    )


## Step 8 — Download runtime assets

Runs the project's own `scripts/download_live_assets.sh` (the authoritative
asset list — now a 10-step download covering the base IMTalker checkpoints,
the lookahead/RMS adapter, the silence-Helium seed, the current 2-hour
static-head LoRA, the cached blink motion, the PersonaPlex bnb4 weights, the
gated PersonaPlex Mimi/tokenizer files, and the `VARM3` voice pack), with
automatic retry on interrupted/failed downloads.


In [ ]:
run(
    f"{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && "
    f"export SPEECH2AVATAR_ROOT={PROJECT_ROOT} && "
    f"bash scripts/download_live_assets.sh",
    cwd=PROJECT_ROOT, retries=3, retry_delay=20,
)


## Step 9 — Checksum verification (LoRA + cached blink motion)

Both files are re-downloaded automatically on checksum mismatch.


In [ ]:
verify_checksum_with_recovery(
    LORA_PATH, HF_LORA_SHA256, HF_LORA_REPO, HF_LORA_FILE, "dataset", CHECKPOINT_DIR,
)
verify_checksum_with_recovery(
    BLINK_MOTION_PATH, HF_BLINK_SHA256, HF_BLINK_REPO, HF_BLINK_FILE, "dataset", CHECKPOINT_DIR,
)


## Step 10 — Verify remaining required assets (incl. the emotion adapter)

Base IMTalker checkpoints, the wav2vec frontend, the lookahead/RMS adapter,
the silence-Helium seed, the PersonaPlex bnb4 weights, the PersonaPlex
Mimi/tokenizer files, the `VARM3` voice prompt (shipped inside `voices.tgz`,
so it is downloaded and extracted separately from the other single-file
assets), and the **emotion LoRA adapter** (`bitwisemind/emotion_adapter_v1` →
`checkpoints/emotion_adapter/best.ckpt`). Anything missing is re-downloaded by
name from the correct repo before the table is reprinted.


In [ ]:
import tarfile

REQUIRED_ASSETS = []
for fname in HF_IMTALKER_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(IMTALKER_CKPT_DIR, fname),
        "repo": HF_IMTALKER_REPO, "filename": fname, "repo_type": "model", "local_dir": IMTALKER_CKPT_DIR,
    })
REQUIRED_ASSETS.append({
    "path": ADAPTER_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_ADAPTER_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": SILENCE_HELIUM_PATH, "repo": HF_ADAPTER_REPO, "filename": HF_SILENCE_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": LORA_PATH, "repo": HF_LORA_REPO, "filename": HF_LORA_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": BLINK_MOTION_PATH, "repo": HF_BLINK_REPO, "filename": HF_BLINK_FILE,
    "repo_type": "dataset", "local_dir": CHECKPOINT_DIR,
})
REQUIRED_ASSETS.append({
    "path": PERSONAPLEX_WEIGHT_PATH, "repo": HF_PERSONAPLEX_REPO, "filename": "model_bnb_4bit.pt",
    "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
})
for fname in HF_PERSONAPLEX_MIMI_FILES:
    REQUIRED_ASSETS.append({
        "path": os.path.join(PERSONAPLEX_BNB4_DIR, fname),
        "repo": HF_PERSONAPLEX_MIMI_REPO, "filename": fname, "repo_type": "model", "local_dir": PERSONAPLEX_BNB4_DIR,
    })

for asset in REQUIRED_ASSETS:
    if not os.path.exists(asset["path"]):
        print(f"[fix] missing asset, downloading: {asset['path']}")
        hf_download(asset["repo"], asset["filename"], asset["local_dir"], asset["repo_type"])

# The VARM3 voice prompt ships inside voices.tgz (nvidia/personaplex-7b-v1), not
# as a standalone HF file, so it needs a download + extract step of its own.
if not os.path.exists(VOICE_PROMPT_PATH):
    print(f"[fix] missing voice prompt, downloading + extracting: {HF_VOICES_FILE}")
    hf_download(HF_PERSONAPLEX_MIMI_REPO, HF_VOICES_FILE, PERSONAPLEX_BNB4_DIR, "model")
    voices_tgz = os.path.join(PERSONAPLEX_BNB4_DIR, HF_VOICES_FILE)
    with tarfile.open(voices_tgz) as tf:
        tf.extractall(PERSONAPLEX_BNB4_DIR)

# The emotion LoRA adapter (bitwisemind/emotion_adapter_v1) is a single full
# Lightning checkpoint; try a model repo first, then fall back to a dataset repo.
if not os.path.exists(EMOTION_ADAPTER_PATH) or os.path.getsize(EMOTION_ADAPTER_PATH) == 0:
    print(f"[fix] missing emotion adapter, downloading: {HF_EMOTION_ADAPTER_REPO}/{HF_EMOTION_ADAPTER_FILE}")
    try:
        hf_download(HF_EMOTION_ADAPTER_REPO, HF_EMOTION_ADAPTER_FILE, EMOTION_ADAPTER_DIR, "model")
    except Exception as e:
        print(f"[recovery] model-repo download failed ({e}); trying dataset repo type...")
        hf_download(HF_EMOTION_ADAPTER_REPO, HF_EMOTION_ADAPTER_FILE, EMOTION_ADAPTER_DIR, "dataset")

print("Required asset status:")
all_present = True
for asset in REQUIRED_ASSETS:
    exists = os.path.exists(asset["path"])
    all_present = all_present and exists
    size = f"{os.path.getsize(asset['path']) / (1 << 20):.1f} MB" if exists else "-"
    print(f"  [{'OK' if exists else 'MISSING'}] {asset['path']}  {size}")

voice_exists = os.path.exists(VOICE_PROMPT_PATH)
all_present = all_present and voice_exists
voice_size = f"{os.path.getsize(VOICE_PROMPT_PATH) / (1 << 20):.1f} MB" if voice_exists else "-"
print(f"  [{'OK' if voice_exists else 'MISSING'}] {VOICE_PROMPT_PATH}  {voice_size}")

emotion_exists = os.path.exists(EMOTION_ADAPTER_PATH) and os.path.getsize(EMOTION_ADAPTER_PATH) > 0
all_present = all_present and emotion_exists
emotion_size = f"{os.path.getsize(EMOTION_ADAPTER_PATH) / (1 << 20):.1f} MB" if emotion_exists else "-"
print(f"  [{'OK' if emotion_exists else 'MISSING'}] {EMOTION_ADAPTER_PATH}  {emotion_size}")

if not all_present:
    raise RuntimeError("One or more required assets are still missing after automatic recovery; see table above")

print("[ok] all required runtime assets present")


## Step 10b — Pre-download the auto-emotion SER model

Downloads the speech-emotion-recognition model into the Hugging Face cache
so the server's first startup doesn't stall on a ~1.1 GB download. Skipped
when `AUTO_EMOTION = False`.


In [ ]:
if AUTO_EMOTION:
    run(f'{VENV_ACTIVATE} && export HF_HUB_ENABLE_HF_TRANSFER=1 && hf download {SER_MODEL}',
        retries=3, retry_delay=15)
    print(f"[ok] SER model cached: {SER_MODEL}")
else:
    print("[skip] AUTO_EMOTION is False - no SER model to download")


## Step 11 — Install PersonaPlex / Moshi (`--no-deps`)

Per `live.md`, the Moshi package source ships inside the downloaded
`personaplex_bnb4` weights folder. As a recovery fallback (in case a given
weights snapshot does not bundle it), this also checks the repo's own
`personaplex/moshi` checkout. Either way the install uses `--no-deps` so the
pinned PyTorch stack from Step 4 is never touched.


In [ ]:
moshi_candidates = [
    os.path.join(PERSONAPLEX_BNB4_DIR, "moshi"),
    os.path.join(PERSONAPLEX_DIR, "moshi"),
]
moshi_dir = next((d for d in moshi_candidates if os.path.isdir(d)), None)
if moshi_dir is None:
    raise RuntimeError(
        f"Could not find a moshi/ package source in any of: {moshi_candidates}. "
        f"Re-run Step 8 download, or confirm the PersonaPlex bnb4 snapshot bundles moshi/."
    )
if moshi_dir != moshi_candidates[0]:
    print(f"[recovery] moshi/ not found under personaplex_bnb4; using fallback source at {moshi_dir}")

run(f"{VENV_ACTIVATE} && pip install -e {moshi_dir} --no-deps", retries=2, retry_delay=10)
assert_torch_pin(auto_fix=True)
print("[ok] PersonaPlex/Moshi installed without modifying the pinned torch stack")


## Step 12 — Inject emotion support into the cloned repos (idempotent)

The stock new-pipeline repo ships **without** emotion conditioning, but the
emotion adapter was trained against emotion-aware code. This cell patches the
cloned repo so the training-time and inference-time architectures match
exactly:

| File | What is injected |
|---|---|
| `IMTalker/generator/FM.py` | `EmotionConditioner` module (emotion id → 7-dim one-hot → 32-dim bias), `emotion_conditioner` submodule on `FMGenerator`, `emotion_id` plumbed through `forward` / `sample` — **byte-identical to `train_emotion_lora.ipynb`** — **plus** `emotion_cfg_scale` plumbing in `sample()` (streaming-only, `EMOTION_PATCH_V3`) |
| `IMTalker/generator/FMT.py` | `emotion_bias_now` / `emotion_bias_prev` kwargs on the transformer `forward`, added to the conditioning sum — **byte-identical to `train_emotion_lora.ipynb`** — **plus** a 3-way **compositional CFG** branch in `forward_with_cfg` (streaming-only, `EMOTION_CFG_V1`) |
| `IMTalker/generator/train_lora.py` | `apply_emotion_lora_to_model()` (targets the same attention/MLP layers as the style LoRA, plus the pose/audio projections) — **byte-identical to `train_emotion_lora.ipynb`** |
| `IMTalker/liveTryHelium...AHAudioPace.py` | `--emotion_id` / `--emotion_lora_path` / `--emotion_lora_rank` / `--emotion_lora_alpha` / `--emotion_cfg_scale` args; emotion-aware `_load_fm` (merge style LoRA → apply emotion LoRA → load emotion ckpt); fixed `emotion_id` in every `fm.sample()` call |
| `IMTalker/start_winner_live.sh` | `EMOTION_ARGS` array built from `EMOTION_ID` / `EMOTION_LORA_PATH` / `EMOTION_LORA_RANK` / `EMOTION_LORA_ALPHA` / `EMOTION_CFG_SCALE` env vars, passed to the server |

**Why the compositional CFG is needed (the "always neutral" fix).** The emotion
adapter was trained and validated at `a_cfg_scale=3.0`, where the emotion delta
(target-emotion bias − null-emotion bias) is amplified 3×. Streaming, however,
keeps the audio CFG at the AH lip-sync-tuned default (~1.13), which also scaled
the emotion delta by only ~1.13 — too weak to move the face, so the avatar looked
neutral no matter which emotion was selected. The patched `forward_with_cfg`
splits guidance into **three** branches — `[null audio + null emotion]`,
`[audio + null emotion]`, `[audio + target emotion]` — and combines them as
`uncond + a_cfg_scale·(audio_cond − uncond) + emotion_cfg_scale·(full_cond − audio_cond)`.
Audio (lip-sync) guidance stays at `a_cfg_scale` while the emotion delta gets its
own `emotion_cfg_scale` (default **3.0**, matching the validated strength). This
leaves all existing streaming behaviour (lip-sync, motion, latency) untouched and
only strengthens the expression.

**Smoothness / real-time-budget patches (EMOTION_LIVE_PATCH_V4 + AV_SMOOTH_PATCH_V1).**
These streaming-stability fixes keep the GPU producer inside its real-time budget
(one 0.96 s chunk = 24 frames must be produced in <= 960 ms) and stop the slow A/V
drift that otherwise builds up over a session. They **ship in the cloned repo**; the
cell re-applies them **idempotently** (skipping when the clone already carries them,
re-applying only if the base repo ever loses them), so the notebook stays correct
either way:

| File | Fix |
|---|---|
| server | Audio catch-up pacing relaxed to **exactly real time** (`1.0/fps`, was `0.92/fps`) so a backlog no longer ratchets latency upward |
| server | `reply_codes` kept on the GPU in `_step`; the CPU copy only happens in the (now off-by-default) `--dump_motion` path -- removes a per-step GPU->CPU sync |
| server | Frame + audio queue enqueues **batched** to one event-loop round-trip per chunk (was ~48 blocking cross-thread round-trips) |
| server | **Real-time budget watchdog**: logs `OVER-BUDGET chunk#...` with the moshi/helium/fm/render cost breakdown the moment a chunk exceeds budget |
| `start_winner_live.sh` | `--render_sub_batch` 8 -> **24** (one batched render per chunk; `RENDER_SUB_BATCH` override); `--dump_motion` now **off by default** (`DUMP_MOTION=1` to enable) |
| `start_winner_live.sh` | SER runs on the **CPU** by default (`SER_DEVICE`) so the emotion model never competes with the time-critical GPU producer |
| browser worklet | Bounded **emergency audio drop** re-enabled (1200 ms cap) so a true overflow skips ahead instead of drifting forever |
| browser page | `frameStore` **purged on every arrival** (render-cursor + 150-frame cap) so an audio-rebuffer freeze can't balloon browser memory |

Every patch is anchored on unique source text and is **idempotent** — re-running
this cell detects the markers and skips instantly (or restores the pristine file
from its `.bak_pre_emotion` backup and re-patches when the patch version changes),
so it is safe to re-run the notebook top-to-bottom.


In [ ]:
import json
from pathlib import Path

GEN_DIR = Path(IMTALKER_DIR) / "generator"
SERVER_FILE = Path(IMTALKER_DIR) / "liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary_AHAudioPace.py"
START_WINNER_SH = Path(IMTALKER_DIR) / "start_winner_live.sh"

# ---------------------------------------------------------------------------
# auto_emotion.py - automatic user-voice emotion detection (V3). Written
# next to the live server; the server imports it inside try/except, so a
# missing file degrades to the fixed --emotion_id behaviour.
# ---------------------------------------------------------------------------
AUTO_EMOTION_MODULE = r'''"""Automatic user-voice emotion detection for the live avatar (EMOTION_LIVE_PATCH_V3).

Taps the user's microphone PCM (already decoded to 24 kHz int16 in the GPU
producer thread), segments it into utterances with an energy-based VAD, runs a
speech-emotion-recognition (SER) model (emotion2vec+ via funasr by default;
wav2vec2 classification heads remain supported) on each completed utterance
in a background daemon thread, and publishes the detected emotion id for the
FM generator to pick up on its next 0.96 s chunk.

Design rules (mirrors the server's own defensive patterns):
  * feed() runs on the GPU producer thread and must stay in the microsecond
    range -- buffering + VAD only, never inference, never blocking.
  * All SER inference happens on a daemon worker thread with its own CUDA
    stream; a slow or failed inference can never stall the streaming loop.
  * Any failure (model load, inference, feed) degrades to the fixed
    --emotion_id behaviour instead of raising.

Every classified utterance is appended to a CSV log (one file per server
start, <IMTalker dir>/emotion_logs/emotion_log_YYYYMMDD_HHMMSS.csv, override
with IMTALKER_EMOTION_LOG_DIR) recording both the detected emotion and the
emotion actually sent to the avatar model.
"""

from __future__ import annotations

import collections
import csv
import os
import queue
import threading
import time
import traceback
from datetime import datetime
from pathlib import Path

import numpy as np

# Emotion ids must match generator/FM.py EMOTION_MAP and train_emotion_lora.ipynb.
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}

TARGET_SR = 24_000  # server mic PCM rate (Mimi sample rate)
SER_SR = 16_000     # SER models expect 16 kHz (emotion2vec and wav2vec2 alike)

DEFAULT_SER_MODEL = "emotion2vec/emotion2vec_plus_base"

# Fallback label order for wav2vec2-style repos (used only when the model
# config carries no id2label). superb/wav2vec2-base-superb-er: 4 classes
# trained on IEMOCAP (English conversational speech), ~95M params.
# emotion2vec+ models do not use this fallback: their labels come from the
# checkpoint's tokens.txt via funasr (see SpeechEmotionClassifier).
DEFAULT_SER_LABELS = ["neu", "hap", "ang", "sad"]

# SER label name -> avatar emotion id (generator/FM.py EMOTION_MAP).
# Covers the IEMOCAP abbreviations (neu/hap/ang/sad), the full-word labels
# used by e.g. ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
# (angry, calm, disgust, fearful, happy, neutral, sad, surprised) and the
# emotion2vec+ classes (angry, disgusted, fearful, happy, neutral, other,
# sad, surprised -- emotion2vec tokens are bilingual, e.g. "生气/angry",
# and are normalised to their English part before this lookup). "other"/
# "unknown"/"<unk>" have no avatar counterpart and fall back to neutral.
# "disgust" is handled separately via the --ser_disgust_map arg because the
# emotion LoRA was not trained on disgust.
SER_LABEL_TO_EMOTION_ID = {
    "neutral": 0, "neutrality": 0, "calm": 0, "neu": 0, "other": 0,
    "unknown": 0, "<unk>": 0,
    "happy": 1, "happiness": 1, "joy": 1, "excited": 1, "hap": 1, "exc": 1,
    "sad": 2, "sadness": 2,
    "angry": 3, "anger": 3, "frustrated": 3, "ang": 3,
    "surprise": 4, "surprised": 4,
    "fear": 5, "fearful": 5, "fearfulness": 5,
}



# ---------------------------------------------------------------------------
# Energy-based VAD / utterance segmenter
# ---------------------------------------------------------------------------

class EnergyVAD:
    """Splits a continuous mono stream into utterances (zero dependencies).

    State machine with hysteresis:
      * speech starts when a frame's RMS >= rms_threshold
      * an utterance closes after `silence_sec` of sub-threshold audio
      * utterances shorter than `min_speech_sec` are dropped (coughs, "yes")
      * utterances longer than `max_utterance_sec` are split so SER latency
        stays bounded on long monologues
      * `pre_roll_sec` of audio before speech onset is kept so the classifier
        sees the true utterance start
    """

    def __init__(self, rms_threshold: float = 0.02, silence_sec: float = 0.7,
                 min_speech_sec: float = 0.5, max_utterance_sec: float = 12.0,
                 pre_roll_sec: float = 0.25, sample_rate: int = TARGET_SR,
                 frame_sec: float = 0.02) -> None:
        self.sample_rate = int(sample_rate)
        self.frame_sec = float(frame_sec)
        self.frame_len = max(1, int(round(self.sample_rate * self.frame_sec)))
        self.rms_threshold = float(rms_threshold)
        self.silence_frames = max(1, int(round(float(silence_sec) / self.frame_sec)))
        self.min_speech_frames = max(1, int(round(float(min_speech_sec) / self.frame_sec)))
        self.max_utterance_frames = max(
            self.min_speech_frames, int(round(float(max_utterance_sec) / self.frame_sec))
        )
        self.pre_roll_frames = max(0, int(round(float(pre_roll_sec) / self.frame_sec)))
        self.reset()

    def reset(self) -> None:
        self._pending = np.zeros(0, dtype=np.float32)  # partial-frame carry-over
        self._pre_roll: collections.deque = collections.deque(maxlen=self.pre_roll_frames)
        self._speech: list = []
        self._in_speech = False
        self._silence_run = 0    # consecutive sub-threshold frames while in speech
        self._speech_frames = 0  # speech (non-silence) frames in current utterance

    def feed(self, pcm: np.ndarray):
        """Feed float32 mono samples @ sample_rate; return an utterance or None."""
        pcm = np.asarray(pcm, dtype=np.float32).reshape(-1)
        if pcm.size:
            self._pending = np.concatenate([self._pending, pcm])
        utterance = None
        n_frames = self._pending.shape[0] // self.frame_len
        for i in range(n_frames):
            frame = self._pending[i * self.frame_len:(i + 1) * self.frame_len]
            out = self._process_frame(frame)
            if out is not None:
                utterance = out  # keep the last completed utterance of this call
        if n_frames:
            self._pending = self._pending[n_frames * self.frame_len:]
        return utterance

    def _process_frame(self, frame: np.ndarray):
        rms = float(np.sqrt(np.mean(np.square(frame, dtype=np.float32))))
        if rms >= self.rms_threshold:
            if not self._in_speech:
                self._in_speech = True
                self._speech = list(self._pre_roll)
                self._speech_frames = 0
                self._silence_run = 0
            self._speech.append(frame)
            self._speech_frames += 1
            self._silence_run = 0
            if self._speech_frames >= self.max_utterance_frames:
                return self._close_utterance(trim=False)
            return None
        # sub-threshold frame
        if self._in_speech:
            self._speech.append(frame)
            self._silence_run += 1
            if self._silence_run >= self.silence_frames:
                return self._close_utterance(trim=True)
        else:
            self._pre_roll.append(frame)
        return None

    def _close_utterance(self, trim: bool):
        frames = self._speech
        if trim and self._silence_run:
            # Keep a short natural tail (~0.15 s) instead of the full silence run.
            keep_tail = min(self._silence_run, int(round(0.15 / self.frame_sec)))
            frames = frames[:len(frames) - self._silence_run + keep_tail]
        speech_ok = self._speech_frames >= self.min_speech_frames
        self._in_speech = False
        self._speech = []
        self._speech_frames = 0
        self._silence_run = 0
        self._pre_roll.clear()
        if not speech_ok or not frames:
            return None
        return np.concatenate(frames).astype(np.float32, copy=False)


# ---------------------------------------------------------------------------
# Speech emotion recognition (SER) classifier
# ---------------------------------------------------------------------------

class SpeechEmotionClassifier:
    """Speech emotion classifier (16 kHz mono input).

    Supports three checkpoint families:
      * emotion2vec+ repos (the default, e.g. emotion2vec/emotion2vec_plus_base:
        9 classes, ~90M params) -> loaded via funasr.AutoModel with hub="hf"
        (requires the funasr package)
      * standard Wav2Vec2ForSequenceClassification repos (architectures field
        names a *ForSequenceClassification / *ForAudioClassification class,
        e.g. superb/wav2vec2-base-superb-er) -> loaded via
        AutoModelForAudioClassification
      * audeering-style repos (wav2vec2 backbone + mean-pool + linear `head`)
        -> loaded via the small documented EmotionModel wrapper
    """

    def __init__(self, model_name: str, device: str = "cuda", half: bool = False,
                 disgust_map: int = 0) -> None:
        import torch

        self.torch = torch
        self.device = torch.device(device)
        self.half = bool(half) and self.device.type == "cuda"
        self.disgust_map = int(disgust_map)
        self.model_name = str(model_name)
        self.feature_extractor = None
        self.id2label: dict = {}
        self._warned_labels: set = set()

        if self._is_emotion2vec():
            self._load_emotion2vec()
            return

        from transformers import AutoConfig, AutoFeatureExtractor

        config = AutoConfig.from_pretrained(self.model_name)
        self.feature_extractor = AutoFeatureExtractor.from_pretrained(self.model_name)
        architectures = [str(a) for a in (getattr(config, "architectures", None) or [])]

        if any(("ForSequenceClassification" in a) or ("ForAudioClassification" in a)
               for a in architectures):
            self._load_standard()
        else:
            self._load_audeering_style(config)

        if self.half:
            self.model = self.model.half()
        self.model.to(self.device).eval()

        id2label = getattr(self.model.config, "id2label", None) or {}
        self.id2label = {int(k): str(v).lower() for k, v in id2label.items()}
        if not self.id2label:
            self.id2label = {i: n for i, n in enumerate(DEFAULT_SER_LABELS)}

    def _is_emotion2vec(self) -> bool:
        return "emotion2vec" in self.model_name.lower()

    def _load_emotion2vec(self) -> None:
        from funasr import AutoModel

        # hub="hf" resolves the checkpoint from the Hugging Face hub (the
        # emotion2vec/* repos), i.e. the same HF cache the deployment
        # notebook pre-downloads into. disable_update skips funasr's PyPI
        # version check so server startup never stalls on the network.
        # fp16 is not applied to the funasr stack (--ser_half is accepted
        # but ignored for this backend).
        self.model = AutoModel(
            model=self.model_name,
            hub="hf",
            device=str(self.device),
            disable_update=True,
        )
        self.half = False
        # Labels are returned per prediction by funasr (from the checkpoint's
        # tokens.txt); there is no static id2label on the funasr side.
        self.id2label = {}

    def _load_standard(self) -> None:
        from transformers import AutoModelForAudioClassification

        self.model = AutoModelForAudioClassification.from_pretrained(self.model_name)

        def forward_logits(input_values):
            return self.model(input_values=input_values).logits

        self._forward_logits = forward_logits

    def _load_audeering_style(self, config) -> None:
        import torch.nn as nn
        from transformers import Wav2Vec2Model, Wav2Vec2PreTrainedModel

        class EmotionModel(Wav2Vec2PreTrainedModel):
            """Wrapper from the audeering model card: wav2vec2 + mean-pool + linear head."""

            def __init__(self, cfg):
                super().__init__(cfg)
                self.config = cfg
                self.wav2vec2 = Wav2Vec2Model(cfg)
                num_labels = len(getattr(cfg, "id2label", None) or {}) or int(
                    getattr(cfg, "num_labels", len(DEFAULT_SER_LABELS))
                )
                self.head = nn.Linear(cfg.hidden_size, num_labels)
                self.init_weights()

            def forward(self, input_values):
                hidden = self.wav2vec2(input_values).last_hidden_state
                return self.head(hidden.mean(dim=1))

        self.model = EmotionModel.from_pretrained(self.model_name)

        def forward_logits(input_values):
            return self.model(input_values)

        self._forward_logits = forward_logits

    def _label_to_emotion_id(self, label: str) -> int:
        key = str(label).strip().lower()
        if "/" in key:
            # emotion2vec+ tokens are bilingual, e.g. "生气/angry" -> "angry"
            key = key.split("/")[-1].strip()
        if key in ("disgust", "disgusted"):
            return self.disgust_map
        if key in SER_LABEL_TO_EMOTION_ID:
            return SER_LABEL_TO_EMOTION_ID[key]
        if key not in self._warned_labels:
            self._warned_labels.add(key)
            print(f"[AUTO-EMOTION] unknown SER label {label!r}; mapping to neutral", flush=True)
        return 0

    def predict(self, pcm_24k: np.ndarray):
        """pcm_24k: float32 mono @ 24 kHz. Returns (emotion_id, label, confidence)."""
        torch = self.torch
        import torchaudio

        wav = torch.from_numpy(np.asarray(pcm_24k, dtype=np.float32).reshape(1, -1))
        if TARGET_SR != SER_SR:
            wav = torchaudio.functional.resample(wav, TARGET_SR, SER_SR)
        pcm_16k = wav[0].numpy()

        if self._is_emotion2vec():
            return self._predict_emotion2vec(pcm_16k)

        inputs = self.feature_extractor(
            pcm_16k, sampling_rate=SER_SR, return_tensors="pt"
        )
        input_values = inputs["input_values"].to(self.device)
        if self.half:
            input_values = input_values.half()
        with torch.inference_mode():
            logits = self._forward_logits(input_values)
            probs = torch.softmax(logits.float(), dim=-1)[0]
        conf, idx = torch.max(probs, dim=0)
        label = self.id2label.get(int(idx), str(int(idx)))
        return self._label_to_emotion_id(label), label, float(conf)

    def _predict_emotion2vec(self, pcm_16k: np.ndarray):
        """emotion2vec+ path: funasr AutoModel.generate on one utterance.

        Returns the same (emotion_id, label, confidence) triple as the
        wav2vec2 path. funasr's generate accepts a 16 kHz float32 numpy
        array directly; a temp-wav fallback keeps older builds working.
        """
        pcm_16k = np.asarray(pcm_16k, dtype=np.float32)
        try:
            results = self.model.generate(
                pcm_16k,
                fs=SER_SR,
                granularity="utterance",
                extract_embedding=False,
            )
        except Exception:
            # Some funasr builds only accept file inputs: fall back to a
            # temp wav (utterances are <= 12 s, so the write is trivial).
            import tempfile

            import soundfile as sf

            fd, tmp_path = tempfile.mkstemp(suffix=".wav")
            os.close(fd)
            try:
                sf.write(tmp_path, pcm_16k, SER_SR)
                results = self.model.generate(
                    tmp_path,
                    granularity="utterance",
                    extract_embedding=False,
                )
            finally:
                try:
                    os.unlink(tmp_path)
                except OSError:
                    pass
        if not results:
            return 0, "neutral", 0.0
        r = results[0]
        labels = [str(x) for x in (r.get("labels") or [])]
        scores = [float(x) for x in (r.get("scores") or [])]
        if not labels or not scores or len(labels) != len(scores):
            return 0, "neutral", 0.0
        idx = int(np.argmax(scores))
        label = labels[idx]
        return self._label_to_emotion_id(label), label, float(scores[idx])



# ---------------------------------------------------------------------------
# Emotion prediction log (one CSV row per classified utterance)
# ---------------------------------------------------------------------------

class EmotionLogWriter:
    """Appends one CSV row per classified user utterance.

    File: <log_dir>/emotion_log_YYYYMMDD_HHMMSS.csv (one per server start).
    The `session` column increments on every browser connection so individual
    conversations can be told apart inside the file. Writes are flushed per
    row (one row per utterance - durability over batching).
    """

    HEADER = [
        "timestamp", "session", "utterance", "duration_sec",
        "detected_label", "detected_emotion", "confidence", "accepted",
        "previous_emotion", "applied_emotion",
    ]

    def __init__(self, log_dir: Path) -> None:
        self._lock = threading.Lock()
        self.session = 0
        self._utterance = 0
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime("%Y%m%d_%H%M%S") + f"_{int(time.time() * 1000) % 1000:03d}"
        self.path = log_dir / f"emotion_log_{stamp}.csv"
        new_file = not self.path.exists() or self.path.stat().st_size == 0
        self._fh = self.path.open("a", encoding="utf-8", newline="")
        self._writer = csv.writer(self._fh)
        if new_file:
            self._writer.writerow(self.HEADER)
            self._fh.flush()

    def new_session(self) -> None:
        with self._lock:
            self.session += 1
            self._utterance = 0

    def log(self, duration_sec: float, detected_label: str, detected_emotion: str,
            confidence: float, accepted: bool, previous_emotion: str,
            applied_emotion: str) -> None:
        row = [
            datetime.now().isoformat(timespec="milliseconds"),
            self.session,
            self._utterance + 1,
            f"{duration_sec:.2f}",
            detected_label,
            detected_emotion,
            f"{confidence:.3f}",
            "yes" if accepted else "no",
            previous_emotion,
            applied_emotion,
        ]
        with self._lock:
            self._utterance += 1
            self._writer.writerow(row)
            self._fh.flush()

    def close(self) -> None:
        with self._lock:
            try:
                self._fh.close()
            except Exception:
                pass


# ---------------------------------------------------------------------------
# Manager: VAD on the GPU thread, SER on a daemon worker thread
# ---------------------------------------------------------------------------

class AutoEmotionManager:
    """Owns the VAD, the utterance queue, the SER worker and the published id.

    feed() is called from the GPU producer thread for every mic packet and is
    deliberately cheap (resample + RMS + buffering). Completed utterances are
    classified on a daemon worker thread; the result is published as a plain
    int that the FM chunk builder reads once per 0.96 s chunk.
    """

    def __init__(self, args) -> None:
        self.args = args
        self.initial_emotion_id = int(getattr(args, "emotion_id", 0))
        self._emotion_id = self.initial_emotion_id
        self.confidence_threshold = float(getattr(args, "ser_confidence_threshold", 0.5))
        self.vad = EnergyVAD(
            rms_threshold=float(getattr(args, "vad_rms_threshold", 0.02)),
            silence_sec=float(getattr(args, "vad_silence_sec", 0.7)),
            min_speech_sec=float(getattr(args, "vad_min_speech_sec", 0.5)),
            max_utterance_sec=float(getattr(args, "vad_max_utterance_sec", 12.0)),
        )
        self._queue: queue.Queue = queue.Queue(maxsize=4)
        self._lock = threading.Lock()
        self._stop = threading.Event()
        self._worker: threading.Thread | None = None
        self._classifier: SpeechEmotionClassifier | None = None
        self._logger: EmotionLogWriter | None = None
        self.utterances_classified = 0
        self.utterances_dropped = 0
        self.feed_errors = 0

    def start(self) -> None:
        """Load the SER model (may raise -> caller disables auto mode) and
        start the daemon worker thread."""
        model_name = str(getattr(self.args, "ser_model", "") or DEFAULT_SER_MODEL)
        device = str(getattr(self.args, "ser_device", "") or getattr(self.args, "device", "cuda"))
        half = bool(getattr(self.args, "ser_half", False))
        disgust_map = int(getattr(self.args, "ser_disgust_map", 0))
        t0 = time.perf_counter()
        self._classifier = SpeechEmotionClassifier(
            model_name, device=device, half=half, disgust_map=disgust_map
        )
        self._logger = self._make_logger()
        self._worker = threading.Thread(
            target=self._worker_loop, name="auto-emotion-ser", daemon=True
        )
        self._worker.start()
        print(
            f"[AUTO-EMOTION] enabled: model={model_name} device={device} half={half} "
            f"confidence_threshold={self.confidence_threshold:.2f} "
            f"initial_emotion={EMOTION_NAMES.get(self.initial_emotion_id, self.initial_emotion_id)} "
            f"(SER model loaded in {time.perf_counter() - t0:.1f}s)",
            flush=True,
        )
        if self._logger is not None:
            print(f"[AUTO-EMOTION] logging predictions to {self._logger.path}", flush=True)

    @staticmethod
    def _make_logger() -> EmotionLogWriter | None:
        """Create the CSV prediction log. IMTALKER_EMOTION_LOG_DIR overrides the
        default <IMTalker dir>/emotion_logs. Any failure just disables logging."""
        try:
            log_dir = os.environ.get("IMTALKER_EMOTION_LOG_DIR", "").strip()
            base = Path(log_dir) if log_dir else Path(__file__).resolve().parent / "emotion_logs"
            return EmotionLogWriter(base)
        except Exception as exc:
            print(f"[AUTO-EMOTION] emotion log disabled: {exc!r}", flush=True)
            return None

    def feed(self, pcm_i16: np.ndarray, input_sr: int) -> None:
        """GPU-producer-thread entry point. Never raises, never blocks."""
        try:
            pcm = np.asarray(pcm_i16, dtype=np.float32) / 32768.0
            if int(input_sr) != TARGET_SR:
                import torch
                import torchaudio

                wav = torch.from_numpy(pcm).view(1, -1)
                pcm = torchaudio.functional.resample(wav, int(input_sr), TARGET_SR)[0].numpy()
            with self._lock:
                utterance = self.vad.feed(pcm)
            if utterance is None:
                return
            try:
                self._queue.put_nowait(utterance)
            except queue.Full:
                # Drop the OLDEST queued utterance (the newest user speech is
                # what a live conversation cares about), same policy as the
                # server's own mic backlog handling.
                try:
                    self._queue.get_nowait()
                    self._queue.put_nowait(utterance)
                except Exception:
                    self.utterances_dropped += 1
        except Exception as exc:
            self.feed_errors += 1
            if self.feed_errors <= 5:
                print(f"[AUTO-EMOTION] feed error #{self.feed_errors}: {exc!r}", flush=True)


    def _worker_loop(self) -> None:
        while not self._stop.is_set():
            try:
                utterance = self._queue.get(timeout=0.5)
            except queue.Empty:
                continue
            dur = utterance.shape[0] / float(TARGET_SR)
            try:
                emotion_id, label, conf = self._classifier.predict(utterance)
            except Exception as exc:
                print(
                    f"[AUTO-EMOTION] inference failed ({dur:.2f}s utterance): {exc!r}",
                    flush=True,
                )
                continue
            self.utterances_classified += 1
            prev = self._emotion_id
            accepted = conf >= self.confidence_threshold
            if accepted:
                self._emotion_id = int(emotion_id)
                action = "switching" if prev != emotion_id else "confirmed"
            else:
                action = f"below threshold {self.confidence_threshold:.2f}; keeping"
            print(
                f"[AUTO-EMOTION] utterance {dur:.2f}s -> {label} (p={conf:.2f}) "
                f"[{EMOTION_NAMES.get(prev, prev)} -> "
                f"{EMOTION_NAMES.get(self._emotion_id, self._emotion_id)}] {action}",
                flush=True,
            )
            if self._logger is not None:
                try:
                    self._logger.log(
                        duration_sec=dur,
                        detected_label=label,
                        detected_emotion=EMOTION_NAMES.get(emotion_id, str(emotion_id)),
                        confidence=conf,
                        accepted=accepted,
                        previous_emotion=EMOTION_NAMES.get(prev, str(prev)),
                        applied_emotion=EMOTION_NAMES.get(self._emotion_id, str(self._emotion_id)),
                    )
                except Exception as exc:
                    print(f"[AUTO-EMOTION] emotion log write failed: {exc!r}", flush=True)

    def current_emotion_id(self, default: int = 0) -> int:
        return int(self._emotion_id)

    def reset_session(self) -> None:
        """Fresh VAD state per browser session. The published emotion resets
        to the configured --emotion_id (e.g. happy) so every fresh connection
        starts with the intended default expression; the SER re-detects and
        takes over from the user's first utterance."""
        with self._lock:
            self.vad.reset()
            self._emotion_id = self.initial_emotion_id
        while True:
            try:
                self._queue.get_nowait()
            except queue.Empty:
                break
        if self._logger is not None:
            self._logger.new_session()

    def stop(self) -> None:
        self._stop.set()
        if self._logger is not None:
            self._logger.close()


# ---------------------------------------------------------------------------
# Module-level singleton (created once at server startup)
# ---------------------------------------------------------------------------

_MANAGER: AutoEmotionManager | None = None


def init_auto_emotion_manager(args) -> AutoEmotionManager | None:
    """Create and start the singleton. On ANY failure, auto mode is disabled
    and the server keeps the fixed --emotion_id behaviour."""
    global _MANAGER
    if _MANAGER is not None:
        return _MANAGER
    try:
        mgr = AutoEmotionManager(args)
        mgr.start()
    except Exception as exc:
        print(
            f"[AUTO-EMOTION] DISABLED - failed to initialise the SER model; "
            f"falling back to fixed --emotion_id={int(getattr(args, 'emotion_id', 0))}: "
            f"{exc!r}\n{traceback.format_exc()}",
            flush=True,
        )
        _MANAGER = None
        return None
    _MANAGER = mgr
    return _MANAGER


def get_auto_emotion_manager() -> AutoEmotionManager | None:
    return _MANAGER

'''

ae_path = Path(IMTALKER_DIR) / "auto_emotion.py"
if ae_path.exists() and ae_path.read_text(encoding="utf-8") == AUTO_EMOTION_MODULE:
    print("[skip] auto_emotion.py already up to date")
else:
    ae_path.write_text(AUTO_EMOTION_MODULE, encoding="utf-8")
    print(f"[written] {ae_path} ({len(AUTO_EMOTION_MODULE)} chars)")

# ---------------------------------------------------------------------------
# Training-side patches — byte-identical to train_emotion_lora.ipynb so the
# inference architecture matches what the emotion adapter was trained against.
# ---------------------------------------------------------------------------
FM_PATCHES = json.loads(r'''[["from generator.wav2vec2 import Wav2VecModel\n", "from generator.wav2vec2 import Wav2VecModel\n\n\n# EMOTION_PATCH_V2\nEMOTION_MAP = {\n    \"neutral\": 0, \"neutrality\": 0,\n    \"happy\": 1, \"happiness\": 1,\n    \"sad\": 2, \"sadness\": 2,\n    \"angry\": 3, \"anger\": 3,\n    \"surprise\": 4, \"surprised\": 4,\n    \"fear\": 5, \"fearful\": 5,\n}\nEMOTION_NAMES = {0: \"neutral\", 1: \"happy\", 2: \"sad\", 3: \"angry\", 4: \"surprise\", 5: \"fear\"}\nNUM_EMOTIONS = 6  # real emotion classes (0-5); index 6 = null CFG token\n\n\nclass EmotionConditioner(nn.Module):\n    \"\"\"\n    Maps a discrete emotion ID to a 7-d one-hot vector (using existing dim_e=7\n    from base_options.py), projects it to dim_c=32, and expands frame-wise.\n\n    At training time, 10% of samples have their emotion replaced with the null\n    token (index 6 = dim_e-1) for classifier-free guidance. At inference time,\n    emotion_id is a hardcoded integer (0-5) set once per conversation.\n\n    Zero-init of the projection linear weight ensures the model starts with\n    neutral (zero emotion bias) and degrades gracefully to pretrained behavior.\n    \"\"\"\n    def __init__(self, num_emotions: int = NUM_EMOTIONS,\n                 dim_e: int = 7, dim_c: int = 32,\n                 dropout_prob: float = 0.1):\n        super().__init__()\n        self.num_emotions = int(num_emotions)\n        self.null_idx = int(dim_e) - 1   # index 6 = null token\n        self.dropout_prob = float(dropout_prob)\n\n        # Same pattern as audio_projection / pose_projection / gaze_projection\n        self.emotion_projection = nn.Sequential(\n            nn.Linear(int(dim_e), int(dim_c)),\n            nn.LayerNorm(int(dim_c)),\n            nn.SiLU(),\n        )\n        # Small normal init: provides immediate non-zero gradients from batch 1\n        # without dead zones, while remaining subtle (std=0.02) to preserve pretrained base quality.\n        nn.init.normal_(self.emotion_projection[0].weight, mean=0.0, std=0.02)\n        nn.init.zeros_(self.emotion_projection[0].bias)\n\n    def null_bias(self, B: int, T: int, device: torch.device) -> torch.Tensor:\n        \"\"\"Returns the null-token embedding [B, T, dim_c] for unconditional CFG.\"\"\"\n        null_ids = torch.full((B,), self.null_idx, dtype=torch.long, device=device)\n        one_hot = F.one_hot(null_ids, num_classes=self.null_idx + 1).float()\n        emb = self.emotion_projection(one_hot)\n        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()\n\n    def forward(self, emotion_id: torch.Tensor, T: int,\n                train: bool = False) -> torch.Tensor:\n        \"\"\"\n        Args:\n            emotion_id : [B] long tensor, values in 0 .. num_emotions-1\n            T          : number of time frames to expand to\n            train      : if True, apply CFG dropout with self.dropout_prob\n        Returns:\n            [B, T, dim_c]  \u2013 additive bias for the conditioning sum\n        \"\"\"\n        B = emotion_id.shape[0]\n        if train and self.dropout_prob > 0.0:\n            null_mask = torch.rand(B, device=emotion_id.device) < self.dropout_prob\n            emotion_id = torch.where(\n                null_mask,\n                torch.full_like(emotion_id, self.null_idx),\n                emotion_id,\n            )\n        # One-hot to dim_e=7\n        one_hot = F.one_hot(emotion_id.long(),\n                            num_classes=self.null_idx + 1).float()  # [B, 7]\n        emb = self.emotion_projection(one_hot)                       # [B, dim_c]\n        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()       # [B, T, dim_c]\n\n\nclass AudioBridge768(nn.Module):\n    \"\"\"Mimi in_dim -> 768 via a shallow MLP.\n\n    Intentionally no output LayerNorm: the downstream audio_projection was\n    trained on raw wav2vec last_hidden_state, which is not unit-normalised\n    per frame. An output LN here would force the bridge into an OOD\n    distribution for the frozen audio_projection.\n    \"\"\"\n\n    def __init__(self, in_dim: int = 512, out_dim: int = 768, hidden_dim: int = 1024):\n        super().__init__()\n        self.in_ln = nn.LayerNorm(in_dim)\n        self.fc1 = nn.Linear(in_dim, hidden_dim)\n        self.act = nn.GELU()\n        self.fc2 = nn.Linear(hidden_dim, out_dim)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.in_ln(x)\n        return self.fc2(self.act(self.fc1(x)))\n\n\nclass AudioTemporalConv32(nn.Module):\n    \"\"\"Mimi in_dim -> 32 with local temporal context.\"\"\"\n\n    def __init__(\n        self,\n        in_dim: int = 512,\n        out_dim: int = 32,\n        channels: int = 256,\n        kernel: int = 5,\n    ):\n        super().__init__()\n        pad = kernel // 2\n        self.in_ln = nn.LayerNorm(in_dim)\n        self.fc_in = nn.Linear(in_dim, channels)\n        self.conv1 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)\n        self.conv2 = nn.Conv1d(channels, channels, kernel_size=kernel, padding=pad, groups=8)\n        self.act = nn.GELU()\n        self.fc_out = nn.Linear(channels, out_dim)\n        self.out_ln = nn.LayerNorm(out_dim)\n        self.out_act = nn.SiLU()\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.in_ln(x)\n        x = self.fc_in(x)\n        x = x.transpose(1, 2)\n        x = self.act(self.conv1(x))\n        x = self.act(self.conv2(x))\n        x = x.transpose(1, 2)\n        x = self.fc_out(x)\n        return self.out_act(self.out_ln(x))\n\n"], ["        self.fmt = FlowMatchingTransformer(opt)\n", "        self.fmt = FlowMatchingTransformer(opt)\n\n        # Emotion conditioning: discrete emotion id -> additive dim_c bias\n        self.emotion_conditioner = EmotionConditioner(\n            num_emotions=NUM_EMOTIONS,\n            dim_e=getattr(opt, \"dim_e\", 7),\n            dim_c=opt.dim_c,\n            dropout_prob=getattr(opt, \"emotion_dropout_prob\", 0.1),\n        )\n"], ["        cam, prev_cam = batch[\"cam_now\"], batch[\"cam_prev\"]\n", "        cam, prev_cam = batch[\"cam_now\"], batch[\"cam_prev\"]\n\n        emotion_id = batch.get(\"emotion_id\", None)\n"], ["        pred = self.fmt(\n", "        # Emotion bias (additive, dim_c space); CFG dropout while training\n        emotion_bias_now = None\n        emotion_bias_prev = None\n        if emotion_id is not None:\n            emotion_bias_now = self.emotion_conditioner(\n                emotion_id, self.num_frames_for_clip, train=self.training\n            )\n            emotion_bias_prev = self.emotion_conditioner(\n                emotion_id, self.num_prev_frames, train=self.training\n            )\n\n        pred = self.fmt(\n"], ["            train=self.training,\n        )", "            train=self.training,\n            emotion_bias_now=emotion_bias_now,\n            emotion_bias_prev=emotion_bias_prev,\n        )"], ["        ref_x = data[\"ref_x\"].to(self._device())\n", "        ref_x = data[\"ref_x\"].to(self._device())\n        # Emotion conditioning for sample(): accept int or tensor emotion_id\n        _emotion_id_raw = data.get(\"emotion_id\", None)\n        if _emotion_id_raw is not None:\n            if isinstance(_emotion_id_raw, int):\n                emotion_id = torch.tensor([_emotion_id_raw], dtype=torch.long, device=self._device())\n            else:\n                emotion_id = torch.as_tensor(_emotion_id_raw, dtype=torch.long, device=self._device())\n        else:\n            emotion_id = None\n        # SILENCE_EMOTION_PATCH_V1: optional speaking-state weight (1.0 =\n        # fully speaking, 0.0 = fully silent) and smooth emotion-switch blend\n        # (emotion_id_prev + emotion_blend). Absent keys keep legacy behaviour.\n        _speaking_weight_raw = data.get(\"speaking_weight\", None)\n        speaking_weight = 1.0 if _speaking_weight_raw is None else min(1.0, max(0.0, float(_speaking_weight_raw)))\n        _emotion_id_prev_raw = data.get(\"emotion_id_prev\", None)\n        if _emotion_id_prev_raw is not None:\n            if isinstance(_emotion_id_prev_raw, int):\n                emotion_id_prev = torch.tensor([_emotion_id_prev_raw], dtype=torch.long, device=self._device())\n            else:\n                emotion_id_prev = torch.as_tensor(_emotion_id_prev_raw, dtype=torch.long, device=self._device())\n        else:\n            emotion_id_prev = None\n        _emotion_blend_raw = data.get(\"emotion_blend\", None)\n        emotion_blend = 1.0 if _emotion_blend_raw is None else min(1.0, max(0.0, float(_emotion_blend_raw)))\n"], ["        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)\n", "        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)\n\n        # Emotion bias for CFG sampling (target emotion + null token)\n        emotion_bias_now = None\n        emotion_bias_prev = None\n        null_emotion_bias_now = None\n        null_emotion_bias_prev = None\n        if emotion_id is not None:\n            emotion_bias_now = self.emotion_conditioner(emotion_id, T, train=False)\n            emotion_bias_prev = self.emotion_conditioner(emotion_id, self.num_prev_frames, train=False)\n            # SILENCE_EMOTION_PATCH_V1: smooth emotion switches -- while a switch\n            # is in flight, blend the displayed (previous) emotion embedding\n            # into the target so the expression morphs instead of popping.\n            if emotion_id_prev is not None and emotion_blend < 1.0:\n                _prev_bias_now = self.emotion_conditioner(emotion_id_prev, T, train=False)\n                _prev_bias_prev = self.emotion_conditioner(emotion_id_prev, self.num_prev_frames, train=False)\n                emotion_bias_now = (1.0 - emotion_blend) * _prev_bias_now + emotion_blend * emotion_bias_now\n                emotion_bias_prev = (1.0 - emotion_blend) * _prev_bias_prev + emotion_blend * emotion_bias_prev\n            null_emotion_bias_now = self.emotion_conditioner.null_bias(B, T, device)\n            null_emotion_bias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device)\n"], ["            def sample_chunk(tt, zt):\n", "            # Per-chunk emotion bias slices (mirrors a_t/gaze_t slicing)\n            if emotion_bias_now is not None:\n                _ebias_now = emotion_bias_now[:, start_idx:end_idx]\n                _null_ebias_now = null_emotion_bias_now[:, start_idx:end_idx]\n                if _ebias_now.shape[1] < self.num_frames_for_clip:\n                    _pad = self.num_frames_for_clip - _ebias_now.shape[1]\n                    _ebias_now = torch.cat([_ebias_now, _ebias_now[:, -1:, :].expand(-1, _pad, -1)], dim=1)\n                    _null_ebias_now = torch.cat([_null_ebias_now, _null_ebias_now[:, -1:, :].expand(-1, _pad, -1)], dim=1)\n                _ebias_prev = emotion_bias_prev\n                _null_ebias_prev = null_emotion_bias_prev\n            else:\n                _ebias_now = _ebias_prev = _null_ebias_now = _null_ebias_prev = None\n\n            def sample_chunk(tt, zt):\n"], ["                    a_cfg_scale=a_cfg_scale,\n                )\n                return out[:, self.num_prev_frames :]", "                    a_cfg_scale=a_cfg_scale,\n                    emotion_bias_now=_ebias_now,\n                    emotion_bias_prev=_ebias_prev,\n                    null_emotion_bias_now=_null_ebias_now,\n                    null_emotion_bias_prev=_null_ebias_prev,\n                )\n                return out[:, self.num_prev_frames :]"], ["            null_emotion_bias_now = self.emotion_conditioner.null_bias(B, T, device)\n            null_emotion_bias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device)\n", "            null_emotion_bias_now = self.emotion_conditioner.null_bias(B, T, device)\n            null_emotion_bias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device)\n        # EMOTION_PATCH_V3: compositional emotion guidance for streaming.\n        # The adapter was validated offline at a_cfg_scale=3.0; streaming keeps\n        # a_cfg_scale ~1.13 (AH-tuned for lip-sync), so the emotion delta gets\n        # its own CFG scale via forward_with_cfg(emotion_cfg_scale=...).\n        _emotion_cfg_scale = getattr(self.opt, \"emotion_cfg_scale\", None)\n        _emotion_cfg_scale = None if _emotion_cfg_scale is None else float(_emotion_cfg_scale)\n        # EMOTION_PATCH_V4: per-emotion CFG boost. The happy smile is mouth-centric\n        # and competes with the lip-synced mouth (driven by the audio CFG), so it\n        # renders weaker than brow/eye-centric emotions (e.g. angry) at the same\n        # scale. Boost happy so it stays clearly visible (--emotion_cfg_boost_happy).\n        if _emotion_cfg_scale is not None and emotion_id is not None:\n            _eid = int(emotion_id.reshape(-1)[0].item()) if torch.is_tensor(emotion_id) else int(emotion_id)\n            if _eid == 1:  # happy\n                _emotion_cfg_scale = _emotion_cfg_scale * float(getattr(self.opt, \"emotion_cfg_boost_happy\", 1.0) or 1.0)\n        # SILENCE_EMOTION_PATCH_V1: speaking-state-aware CFG rebalance. The\n        # emotion bias itself stays in the conditioning in BOTH states (the\n        # expression is retained during silence); only the guidance balance\n        # changes. While speaking (speaking_weight -> 1) both scales keep their\n        # speech values (a_cfg_scale for lip-sync, emotion_cfg_scale for the\n        # expression). While silent (speaking_weight -> 0) the audio guidance\n        # rises to a_cfg_scale_silence so the silence-seed features anchor the\n        # mouth closed, and the emotion guidance relaxes to\n        # emotion_cfg_scale_silence so the retained expression no longer drags\n        # the lips. Values in between give smooth gate transitions.\n        _emotion_cfg_silence = getattr(self.opt, \"emotion_cfg_scale_silence\", None)\n        _emotion_cfg_silence = None if _emotion_cfg_silence is None else float(_emotion_cfg_silence)\n        if _emotion_cfg_scale is not None and _emotion_cfg_silence is not None and speaking_weight < 1.0:\n            _emotion_cfg_scale = _emotion_cfg_silence + (_emotion_cfg_scale - _emotion_cfg_silence) * speaking_weight\n        _a_cfg_silence = getattr(self.opt, \"a_cfg_scale_silence\", None)\n        if _a_cfg_silence is not None and speaking_weight < 1.0:\n            _a_cfg_silence = float(_a_cfg_silence)\n            a_cfg_scale = _a_cfg_silence + (float(a_cfg_scale) - _a_cfg_silence) * speaking_weight\n"], ["                    a_cfg_scale=a_cfg_scale,\n                    emotion_bias_now=_ebias_now,\n                    emotion_bias_prev=_ebias_prev,\n                    null_emotion_bias_now=_null_ebias_now,\n                    null_emotion_bias_prev=_null_ebias_prev,\n                )\n", "                    a_cfg_scale=a_cfg_scale,\n                    emotion_bias_now=_ebias_now,\n                    emotion_bias_prev=_ebias_prev,\n                    null_emotion_bias_now=_null_ebias_now,\n                    null_emotion_bias_prev=_null_ebias_prev,\n                    emotion_cfg_scale=_emotion_cfg_scale,\n                )\n"]]''')

FMT_PATCHES = json.loads(r'''[["        train: bool = True,\n        **kwargs\n    ) -> torch.Tensor:", "        train: bool = True,\n        emotion_bias_now=None,\n        emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:"], ["        c = self.c_embedder(a + pose + cam + gaze)\n", "        # Build conditioning sum; add emotion bias BEFORE c_embedder projection\n        conditioning_sum = a + pose + cam + gaze\n        if emotion_bias_now is not None:\n            T_total = conditioning_sum.shape[1]\n            if emotion_bias_prev is not None:\n                full_emotion_bias = torch.cat([emotion_bias_prev, emotion_bias_now], dim=1)\n            else:\n                full_emotion_bias = emotion_bias_now\n            if full_emotion_bias.shape[1] != T_total:\n                full_emotion_bias = F.interpolate(\n                    full_emotion_bias.transpose(1, 2),\n                    size=T_total, mode='nearest'\n                ).transpose(1, 2)\n            conditioning_sum = conditioning_sum + full_emotion_bias\n        c = self.c_embedder(conditioning_sum)\n"], ["        a_cfg_scale: float = 1.0,\n        **kwargs\n    ) -> torch.Tensor:", "        a_cfg_scale: float = 1.0,\n        emotion_bias_now=None,\n        emotion_bias_prev=None,\n        null_emotion_bias_now=None,\n        null_emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:"], ["            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)\n\n            model_output = self.forward(", "            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)\n\n            # Emotion bias: uncond branch uses null token; cond branch uses target emotion\n            if emotion_bias_now is not None:\n                uncond_now = null_emotion_bias_now if null_emotion_bias_now is not None else emotion_bias_now\n                ebias_now_cat = torch.cat([uncond_now, emotion_bias_now], dim=0)\n            else:\n                ebias_now_cat = None\n\n            if emotion_bias_prev is not None:\n                uncond_prev = null_emotion_bias_prev if null_emotion_bias_prev is not None else emotion_bias_prev\n                ebias_prev_cat = torch.cat([uncond_prev, emotion_bias_prev], dim=0)\n            else:\n                ebias_prev_cat = None\n\n            model_output = self.forward("], ["                prev_cam=prev_cam_cat,\n                train=False\n            )", "                prev_cam=prev_cam_cat,\n                train=False,\n                emotion_bias_now=ebias_now_cat,\n                emotion_bias_prev=ebias_prev_cat,\n            )"], ["                prev_cam=prev_cam,\n                train=False\n            )", "                prev_cam=prev_cam,\n                train=False,\n                emotion_bias_now=emotion_bias_now,\n                emotion_bias_prev=emotion_bias_prev,\n            )"], ["        null_emotion_bias_now=None,\n        null_emotion_bias_prev=None,\n        **kwargs\n    ) -> torch.Tensor:\n        if a_cfg_scale != 1.0:\n", "        null_emotion_bias_now=None,\n        null_emotion_bias_prev=None,\n        emotion_cfg_scale=None,\n        **kwargs\n    ) -> torch.Tensor:\n        # EMOTION_CFG_V1: compositional guidance - independent audio/emotion CFG.\n        # Streaming keeps a_cfg_scale low (~1.13, AH-tuned for lip-sync) while the\n        # emotion delta gets its own scale (adapter was validated at cfg=3.0).\n        if emotion_cfg_scale is not None and emotion_bias_now is not None:\n            null_a_e = torch.zeros_like(a)\n            unc_now = null_emotion_bias_now if null_emotion_bias_now is not None else emotion_bias_now\n            if emotion_bias_prev is not None:\n                unc_prev = null_emotion_bias_prev if null_emotion_bias_prev is not None else emotion_bias_prev\n            else:\n                unc_prev = None\n            if a_cfg_scale != 1.0:\n                # 3 branches: [null audio + null emotion, audio + null emotion, audio + target emotion]\n                if unc_prev is not None:\n                    ebias_prev_cat = torch.cat([unc_prev, unc_prev, emotion_bias_prev], dim=0)\n                else:\n                    ebias_prev_cat = None\n                model_output = self.forward(\n                    t=t,\n                    x=torch.cat([x, x, x], dim=0),\n                    a=torch.cat([null_a_e, a, a], dim=0),\n                    prev_x=torch.cat([prev_x, prev_x, prev_x], dim=0),\n                    prev_a=torch.cat([prev_a, prev_a, prev_a], dim=0),\n                    ref_x=torch.cat([ref_x, ref_x, ref_x], dim=0),\n                    gaze=torch.cat([gaze, gaze, gaze], dim=0),\n                    prev_gaze=torch.cat([prev_gaze, prev_gaze, prev_gaze], dim=0),\n                    pose=torch.cat([pose, pose, pose], dim=0),\n                    prev_pose=torch.cat([prev_pose, prev_pose, prev_pose], dim=0),\n                    cam=torch.cat([cam, cam, cam], dim=0),\n                    prev_cam=torch.cat([prev_cam, prev_cam, prev_cam], dim=0),\n                    train=False,\n                    emotion_bias_now=torch.cat([unc_now, unc_now, emotion_bias_now], dim=0),\n                    emotion_bias_prev=ebias_prev_cat,\n                )\n                uncond, audio_cond, full_cond = torch.chunk(model_output, chunks=3, dim=0)\n                return uncond + a_cfg_scale * (audio_cond - uncond) + emotion_cfg_scale * (full_cond - audio_cond)\n            # 2 branches: [audio + null emotion, audio + target emotion]\n            if unc_prev is not None:\n                ebias_prev_cat = torch.cat([unc_prev, emotion_bias_prev], dim=0)\n            else:\n                ebias_prev_cat = None\n            model_output = self.forward(\n                t=t,\n                x=torch.cat([x, x], dim=0),\n                a=torch.cat([a, a], dim=0),\n                prev_x=torch.cat([prev_x, prev_x], dim=0),\n                prev_a=torch.cat([prev_a, prev_a], dim=0),\n                ref_x=torch.cat([ref_x, ref_x], dim=0),\n                gaze=torch.cat([gaze, gaze], dim=0),\n                prev_gaze=torch.cat([prev_gaze, prev_gaze], dim=0),\n                pose=torch.cat([pose, pose], dim=0),\n                prev_pose=torch.cat([prev_pose, prev_pose], dim=0),\n                cam=torch.cat([cam, cam], dim=0),\n                prev_cam=torch.cat([prev_cam, prev_cam], dim=0),\n                train=False,\n                emotion_bias_now=torch.cat([unc_now, emotion_bias_now], dim=0),\n                emotion_bias_prev=ebias_prev_cat,\n            )\n            audio_cond, full_cond = torch.chunk(model_output, chunks=2, dim=0)\n            return audio_cond + emotion_cfg_scale * (full_cond - audio_cond)\n        if a_cfg_scale != 1.0:\n"]]''')

TRAIN_LORA_APPEND = r'''def apply_emotion_lora_to_model(
    model,
    rank: int = 32,
    alpha: float = None,
    dropout: float = 0.05,
):
    """
    Emotion-specific LoRA fine-tuning setup.

    Freezes ALL base model parameters, then:
    1. Unfreezes the emotion_conditioner entirely (new, small module).
    2. Applies LoRA to: fmt.c_embedder, all fmt.blocks.*.adaLN_modulation.1,
       fmt.decoder.linear, fmt.decoder.adaLN_modulation.1.

    These are exactly the layers that control how conditioning signals
    (now including emotion) modulate each transformer block.
    Audio/pose/gaze/cam projections are frozen to preserve lip-sync quality.
    """
    if alpha is None or float(alpha) <= 0:
        alpha = float(2 * int(rank))

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze emotion_conditioner (new module, not in pretrained ckpt)
    for param in model.emotion_conditioner.parameters():
        param.requires_grad = True

    # Collect LoRA targets across transformer blocks, conditioning embedder, and decoder
    targets = []
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        is_attn_mlp = name.startswith("fmt.blocks.") and (
            name.endswith(".attn.qkv")
            or name.endswith(".attn.proj")
            or name.endswith(".mlp.fc1")
            or name.endswith(".mlp.fc2")
        )
        is_adaLN = (
            name.startswith("fmt.blocks.")
            and name.endswith(".adaLN_modulation.1")
        )
        is_decoder = name in {
            "fmt.decoder.linear",
            "fmt.decoder.adaLN_modulation.1",
        }
        is_c_embedder = name == "fmt.c_embedder"
        if is_attn_mlp or is_adaLN or is_decoder or is_c_embedder:
            targets.append(name)

    for name in targets:
        base = _get_child(model, name)
        _set_child(
            model, name,
            LoRALinear(base, rank=int(rank), alpha=float(alpha), dropout=float(dropout))
        )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(
        f"[EmotionLoRA] targets={len(targets)} rank={rank} alpha={alpha} dropout={dropout}",
        flush=True,
    )
    print(
        f"[EmotionLoRA] trainable={trainable:,} total={total:,} "
        f"ratio={trainable / max(1, total):.6f}",
        flush=True,
    )
    return targets'''

# ---------------------------------------------------------------------------
# Live-serving patches — add --emotion_* args, emotion-aware _load_fm (merge
# style LoRA -> apply emotion LoRA -> load emotion ckpt), fixed emotion_id in
# every fm.sample() call, and EMOTION_ARGS plumbing in start_winner_live.sh.
# ---------------------------------------------------------------------------
SERVER_PATCHES = json.loads(r'''[["        parser.add_argument(\"--lora_generator_path\", default=\"\", help=\"Optional LoRA generator checkpoint to apply on top of --generator_path\")\n", "        parser.add_argument(\"--lora_generator_path\", default=\"\", help=\"Optional LoRA generator checkpoint to apply on top of --generator_path\")\n        # EMOTION_LIVE_PATCH_V1\n        parser.add_argument(\"--emotion_id\", type=int, default=0,\n                            help=\"Hardcoded emotion for the avatar: 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear\")\n        parser.add_argument(\"--emotion_lora_path\", default=\"\",\n                            help=\"Path to emotion LoRA checkpoint (stacked on top of --lora_generator_path)\")\n        parser.add_argument(\"--emotion_lora_rank\",  type=int,   default=32)\n        parser.add_argument(\"--emotion_lora_alpha\", type=float, default=64.0)\n        # EMOTION_LIVE_PATCH_V2\n        parser.add_argument(\"--emotion_cfg_scale\", type=float, default=3.0,\n                            help=\"Compositional CFG scale for the emotion delta (audio stays at --a_cfg_scale). 1.0 = no extra emotion guidance\")\n        parser.add_argument(\"--emotion_cfg_boost_happy\", type=float, default=1.0,\n                            help=\"Extra CFG multiplier for the happy emotion (its smile competes with the lip-synced mouth, so it renders weaker than angry). 1.0 = no boost\")\n        # EMOTION_LIVE_PATCH_V3 (auto emotion from the user's voice)\n        parser.add_argument(\"--auto_emotion\", action=\"store_true\",\n                            help=\"Detect the user's emotion from their voice each utterance and drive the avatar's expression automatically (--emotion_id becomes the initial/fallback emotion)\")\n        parser.add_argument(\"--ser_model\", default=\"emotion2vec/emotion2vec_plus_base\",\n                            help=\"Hugging Face speech-emotion-recognition model (emotion2vec+ via funasr, or a wav2vec2 classification head; 16 kHz input)\")\n        parser.add_argument(\"--ser_device\", default=\"\",\n                            help=\"Device for SER inference (default: same as --device; use 'cpu' to avoid GPU contention)\")\n        parser.add_argument(\"--ser_half\", action=\"store_true\",\n                            help=\"Run the SER model in fp16 (halves VRAM)\")\n        parser.add_argument(\"--ser_confidence_threshold\", type=float, default=0.5,\n                            help=\"Minimum softmax probability to accept a detected emotion; below it the previous emotion is kept\")\n        parser.add_argument(\"--ser_disgust_map\", type=int, default=0,\n                            help=\"Emotion id the SER 'disgust' class maps to (0=neutral; 3=angry is the closest trained class)\")\n        parser.add_argument(\"--vad_rms_threshold\", type=float, default=0.02,\n                            help=\"Energy VAD: RMS threshold for speech on the 24kHz mic signal\")\n        parser.add_argument(\"--vad_silence_sec\", type=float, default=0.7,\n                            help=\"Energy VAD: trailing silence that closes an utterance\")\n        parser.add_argument(\"--vad_min_speech_sec\", type=float, default=0.5,\n                            help=\"Energy VAD: minimum speech duration for an utterance to be classified\")\n        parser.add_argument(\"--vad_max_utterance_sec\", type=float, default=12.0,\n                            help=\"Energy VAD: longer utterances are split and classified in pieces\")\n        # SILENCE_EMOTION_PATCH_V1: speaking-state-aware CFG scales. The avatar\n        # gate (assistant_active) already switches the Helium input between live\n        # speech hidden states and the silence_helium_mean seed; these scales let\n        # the FM rebalance its compositional CFG per chunk so the retained\n        # emotion expression stops dragging the lips open while idle.\n        parser.add_argument(\"--a_cfg_scale_silence\", type=float, default=3.0,\n                            help=\"Audio CFG scale while the avatar is NOT speaking (silence-seed driven). \"\n                                 \"Anchors the mouth closed; 3.0 matches the offline-validated adapter CFG.\")\n        parser.add_argument(\"--emotion_cfg_scale_silence\", type=float, default=1.5,\n                            help=\"Emotion CFG scale while the avatar is NOT speaking. Keeps the retained \"\n                                 \"expression visible without the speech-level amplification that moves the lips.\")\n        parser.add_argument(\"--gate_cfg_ramp_chunks\", type=int, default=2,\n                            help=\"Avatar chunks (~0.96s each) used to ramp the CFG scales at speech start/end.\")\n        parser.add_argument(\"--emotion_switch_ramp_chunks\", type=int, default=2,\n                            help=\"Avatar chunks (~0.96s each) used to blend one detected emotion into the next.\")\n"], ["    lora_path = str(getattr(args, \"lora_generator_path\", \"\") or \"\")\n    if lora_path:\n        apply_lora_to_model(\n", "    lora_path = str(getattr(args, \"lora_generator_path\", \"\") or \"\")\n    emotion_lora_path = str(getattr(args, \"emotion_lora_path\", \"\") or \"\")\n    has_emotion_lora = bool(emotion_lora_path and Path(emotion_lora_path).exists())\n    if emotion_lora_path and not has_emotion_lora:\n        print(f\"[liveTryHeliumFM][FM] WARNING: emotion_lora_path not found: {emotion_lora_path}\", flush=True)\n\n    if lora_path and has_emotion_lora:\n        # Both the style LoRA and the emotion LoRA are provided: merge the\n        # style LoRA delta weights directly into the base generator weights\n        # (scale = alpha / rank) so the emotion LoRA (different rank) can be\n        # applied on top without a rank conflict.\n        from generator.train_lora import _get_child, apply_emotion_lora_to_model\n        lora_ckpt = torch.load(lora_path, map_location=\"cpu\")\n        lora_cleaned = _clean_generator_state(lora_ckpt)\n        style_rank = int(getattr(args, \"lora_rank\", 64) or 64)\n        style_alpha = float(getattr(args, \"lora_alpha\", 128) or 128)\n        style_scale = style_alpha / max(1, style_rank)\n        lora_targets = set()\n        for k in lora_cleaned.keys():\n            if \".lora_A.weight\" in k:\n                lora_targets.add(k.replace(\".lora_A.weight\", \"\"))\n        merged = 0\n        for target in lora_targets:\n            A = lora_cleaned.get(f\"{target}.lora_A.weight\")\n            B = lora_cleaned.get(f\"{target}.lora_B.weight\")\n            if A is not None and B is not None:\n                delta_W = style_scale * (B.float() @ A.float())\n                try:\n                    mod = _get_child(fm, target)\n                    mod.weight.data += delta_W.to(device=mod.weight.device, dtype=mod.weight.dtype)\n                    merged += 1\n                except Exception as exc:\n                    print(f\"[liveTryHeliumFM][FM] could not merge style layer {target}: {exc}\", flush=True)\n        print(\n            f\"[liveTryHeliumFM][FM] merged {merged} style LoRA layers from {Path(lora_path).name} \"\n            f\"(rank={style_rank}, scale={style_scale:.2f}) into base generator weights\",\n            flush=True,\n        )\n        # Now apply the emotion LoRA on top of the merged generator\n        apply_emotion_lora_to_model(\n            fm,\n            rank=int(getattr(args, \"emotion_lora_rank\", 32) or 32),\n            alpha=float(getattr(args, \"emotion_lora_alpha\", 64.0) or 64.0),\n            dropout=0.0,\n        )\n        emo_ckpt = torch.load(emotion_lora_path, map_location=\"cpu\", weights_only=False)\n        emo_cleaned = _clean_generator_state(emo_ckpt)\n        missing_emo, unexpected_emo = fm.load_state_dict(emo_cleaned, strict=False)\n        emo_keys = sum(1 for k in emo_cleaned if \"emotion\" in k or \"lora_\" in k)\n        print(\n            f\"[liveTryHeliumFM][FM] emotion lora loaded path={emotion_lora_path} \"\n            f\"emo_keys={emo_keys} missing={len(missing_emo)} unexpected={len(unexpected_emo)}\",\n            flush=True,\n        )\n    elif lora_path:\n        apply_lora_to_model(\n"], ["    fm.to(device).eval()\n", "    elif has_emotion_lora:\n        # Emotion LoRA only (no style LoRA): apply it directly on the base generator\n        from generator.train_lora import apply_emotion_lora_to_model\n        apply_emotion_lora_to_model(\n            fm,\n            rank=int(getattr(args, \"emotion_lora_rank\", 32) or 32),\n            alpha=float(getattr(args, \"emotion_lora_alpha\", 64.0) or 64.0),\n            dropout=0.0,\n        )\n        emo_ckpt = torch.load(emotion_lora_path, map_location=\"cpu\", weights_only=False)\n        emo_cleaned = _clean_generator_state(emo_ckpt)\n        missing_emo, unexpected_emo = fm.load_state_dict(emo_cleaned, strict=False)\n        emo_keys = sum(1 for k in emo_cleaned if \"emotion\" in k or \"lora_\" in k)\n        print(\n            f\"[liveTryHeliumFM][FM] emotion lora loaded path={emotion_lora_path} \"\n            f\"emo_keys={emo_keys} missing={len(missing_emo)} unexpected={len(unexpected_emo)}\",\n            flush=True,\n        )\n    fm.to(device).eval()\n"], ["        data: dict = {\"a_feat\": feat_25.unsqueeze(0).float(), \"ref_x\": self.ref_x}\n", "        # EMOTION_LIVE_PATCH_V3: emotion_id is resolved per chunk. With\n        # --auto_emotion the AutoEmotionManager's latest detection wins;\n        # otherwise the fixed --emotion_id is used (V1/V2 behaviour).\n        _emotion_id = int(getattr(self.args, \"emotion_id\", 0))\n        _auto_emotion_mgr = get_auto_emotion_manager()\n        if _auto_emotion_mgr is not None:\n            _emotion_id = _auto_emotion_mgr.current_emotion_id(default=_emotion_id)\n        # SILENCE_EMOTION_PATCH_V1: speaking-state awareness. The producer passes\n        # speaking=True while the assistant reply drives the avatar and\n        # speaking=False while the silence_helium_mean seed drives it. The FM\n        # keeps the SAME emotion conditioning in both states (the expression\n        # is retained during silence); only the compositional-CFG balance\n        # changes, interpolated by a per-chunk speaking weight: audio guidance\n        # between --a_cfg_scale (speech, lip-sync) and --a_cfg_scale_silence\n        # (silence, mouth anchored closed), emotion guidance between\n        # --emotion_cfg_scale and --emotion_cfg_scale_silence. The weight ramps\n        # over --gate_cfg_ramp_chunks so gate transitions stay smooth.\n        _gate_ramp = max(1, int(getattr(self.args, \"gate_cfg_ramp_chunks\", 2) or 2))\n        _speaking_target = 1.0 if speaking else 0.0\n        _speaking_weight = getattr(self, \"_speaking_weight\", None)\n        if _speaking_weight is None:\n            _speaking_weight = _speaking_target\n        elif _speaking_weight < _speaking_target:\n            _speaking_weight = min(_speaking_target, _speaking_weight + 1.0 / float(_gate_ramp))\n        elif _speaking_weight > _speaking_target:\n            _speaking_weight = max(_speaking_target, _speaking_weight - 1.0 / float(_gate_ramp))\n        self._speaking_weight = float(_speaking_weight)\n        # SILENCE_EMOTION_PATCH_V1: smooth emotion switches. Blend the displayed\n        # emotion into a newly detected one over --emotion_switch_ramp_chunks\n        # instead of hard-switching the conditioning (which pops the face).\n        _eid_ramp = max(1, int(getattr(self.args, \"emotion_switch_ramp_chunks\", 2) or 2))\n        _eid_displayed = getattr(self, \"_eid_displayed\", None)\n        if _eid_displayed is None:\n            _eid_displayed = _emotion_id\n        if _emotion_id != getattr(self, \"_eid_target_prev\", _emotion_id):\n            self._eid_blend = 0.0\n        self._eid_target_prev = int(_emotion_id)\n        _eid_blend = float(getattr(self, \"_eid_blend\", 1.0))\n        if _eid_blend < 1.0 and _emotion_id != _eid_displayed:\n            _eid_blend = min(1.0, _eid_blend + 1.0 / float(_eid_ramp))\n        elif _emotion_id == _eid_displayed:\n            _eid_blend = 1.0\n        self._eid_blend = _eid_blend\n        if _eid_blend >= 1.0:\n            _eid_displayed = _emotion_id\n        self._eid_displayed = int(_eid_displayed)\n        data: dict = {\"a_feat\": feat_25.unsqueeze(0).float(), \"ref_x\": self.ref_x,\n                      \"emotion_id\": _emotion_id,\n                      \"emotion_id_prev\": int(_eid_displayed),\n                      \"emotion_blend\": float(_eid_blend),\n                      \"speaking_weight\": float(_speaking_weight)}\n"], ["    def _sample_motion_from_helium(self, helium: torch.Tensor, target_frames: int) -> tuple[torch.Tensor, dict]:", "    # SILENCE_EMOTION_PATCH_V1: `speaking` tells this chunk whether the avatar\n    # gate is driving motion from live reply hidden states (True) or from the\n    # silence_helium_mean seed (False). It only rebalances the FM CFG; the\n    # silence seed / deque / gate logic itself is unchanged.\n    def _sample_motion_from_helium(self, helium: torch.Tensor, target_frames: int,\n                                   speaking: bool = True) -> tuple[torch.Tensor, dict]:"], ["                            motion, fm_info = fm_engine._sample_motion_from_helium(helium_chunk, target_frames)", "                            motion, fm_info = fm_engine._sample_motion_from_helium(helium_chunk, target_frames, speaking=bool(assistant_active))"], ["        self.helium_context_tail = None\n        self.helium_deque = None\n        self.helium_deque_filled = 0\n        self._pcm_accum = np.empty(0, dtype=np.float32)", "        self.helium_context_tail = None\n        self.helium_deque = None\n        self.helium_deque_filled = 0\n        self._pcm_accum = np.empty(0, dtype=np.float32)\n        # SILENCE_EMOTION_PATCH_V1: fresh speaking/emotion-ramp state so a new\n        # session re-anchors from the current gate/emotion instead of carrying\n        # the previous session ramp position over.\n        self._speaking_weight = None\n        self._eid_displayed = None\n        self._eid_blend = 1.0\n        self._eid_target_prev = None"], ["from renderer.models import IMTRenderer\n", "from renderer.models import IMTRenderer\n\n# EMOTION_LIVE_PATCH_V3: auto emotion detection from the user's voice.\n# auto_emotion.py is written next to this file by the deployment notebook;\n# if it is missing or fails to import, auto mode degrades to the fixed\n# --emotion_id behaviour.\ntry:\n    from auto_emotion import get_auto_emotion_manager, init_auto_emotion_manager\nexcept Exception:  # auto_emotion.py not deployed\n    def get_auto_emotion_manager():\n        return None\n\n    def init_auto_emotion_manager(args):\n        return None\n"], ["    engine: LiveHeliumFMEngine | None = LiveHeliumFMEngine(args)\n    moshi_engine: MoshiOnlyEngine | None = None\n", "    engine: LiveHeliumFMEngine | None = LiveHeliumFMEngine(args)\n    moshi_engine: MoshiOnlyEngine | None = None\n\n    # EMOTION_LIVE_PATCH_V3: load the SER model once at startup when\n    # --auto_emotion is set. Any failure disables auto mode (fixed\n    # --emotion_id fallback) instead of crashing the server.\n    if bool(getattr(args, \"auto_emotion\", False)):\n        init_auto_emotion_manager(args)\n"], ["        fm_engine = get_engine()\n        fm_engine.reset_session()\n        reply_engine = get_moshi_engine() if args.enable_moshi_reply else None\n        if reply_engine is not None:\n            reply_engine.reset_session()\n", "        fm_engine = get_engine()\n        fm_engine.reset_session()\n        reply_engine = get_moshi_engine() if args.enable_moshi_reply else None\n        if reply_engine is not None:\n            reply_engine.reset_session()\n        # EMOTION_LIVE_PATCH_V3: fresh VAD state per browser session.\n        _auto_emotion_mgr = get_auto_emotion_manager()\n        if _auto_emotion_mgr is not None:\n            _auto_emotion_mgr.reset_session()\n"], ["                while not stopped:\n                    # --- Phase 1: drain all available mic audio (non-blocking) ---\n                    while True:\n                        try:\n                            item = mic_q.get_nowait()\n                        except queue.Empty:\n                            break\n                        if item is None:\n                            stopped = True\n                            break\n                        raw_bytes, input_sr = item\n                        if raw_bytes:\n                            reply_engine.append_browser_pcm(\n                                np.frombuffer(raw_bytes, dtype=np.int16), input_sr\n                            )\n", "                # EMOTION_LIVE_PATCH_V3: fetch the auto-emotion manager once per\n                # session (None when --auto_emotion is off or the SER model\n                # failed to load).\n                _auto_emotion_mgr = get_auto_emotion_manager()\n\n                while not stopped:\n                    # --- Phase 1: drain all available mic audio (non-blocking) ---\n                    while True:\n                        try:\n                            item = mic_q.get_nowait()\n                        except queue.Empty:\n                            break\n                        if item is None:\n                            stopped = True\n                            break\n                        raw_bytes, input_sr = item\n                        if raw_bytes:\n                            _pcm_i16 = np.frombuffer(raw_bytes, dtype=np.int16)\n                            reply_engine.append_browser_pcm(_pcm_i16, input_sr)\n                            # EMOTION_LIVE_PATCH_V3: tap the user's mic for auto\n                            # emotion detection (VAD + buffering only; SER runs\n                            # on a daemon thread). Never allowed to raise here.\n                            if _auto_emotion_mgr is not None:\n                                try:\n                                    _auto_emotion_mgr.feed(_pcm_i16, int(input_sr))\n                                except Exception as _ae_exc:\n                                    print(f\"[AUTO-EMOTION] feed failed: {_ae_exc!r}\", flush=True)\n"], ["                min_send_interval_s = max(0.001, 0.92 / float(args.fps))", "                min_send_interval_s = max(0.001, 1.0 / float(args.fps))"], ["            reply_codes = tokens[:, 1:].detach().to(device=\"cpu\", dtype=torch.int16)", "            # Keep reply_codes on the GPU (a cheap view+detach, no sync). The\n            # CPU int16 copy is only needed by the --dump_motion recording path,\n            # which now does the transfer itself -- this removes a per-step\n            # GPU->CPU sync (12.5x/sec) from the hot path when not dumping.\n            reply_codes = tokens[:, 1:].detach()"], ["                            used_codes = [\n                                s[\"reply_codes\"].to(dtype=torch.int16).contiguous()\n                                for s in used_steps\n                                if isinstance(s.get(\"reply_codes\"), torch.Tensor)\n                            ]\n                            if used_codes and fm_engine.dump_motion:\n                                fm_engine._session_live_token_parts.extend(used_codes)\n                            fm_engine._record_session_chunk(pcm_chunk, motion, fm_info)", "                            if fm_engine.dump_motion:\n                                used_codes = [\n                                    s[\"reply_codes\"].detach().to(device=\"cpu\", dtype=torch.int16).contiguous()\n                                    for s in used_steps\n                                    if isinstance(s.get(\"reply_codes\"), torch.Tensor)\n                                ]\n                                if used_codes:\n                                    fm_engine._session_live_token_parts.extend(used_codes)\n                            fm_engine._record_session_chunk(pcm_chunk, motion, fm_info)"], ["                            used_codes = [\n                                s[\"reply_codes\"].to(dtype=torch.int16).contiguous()\n                                for s in used_steps\n                                if isinstance(s.get(\"reply_codes\"), torch.Tensor)\n                            ]\n                            if used_codes and fm_engine.dump_motion:\n                                fm_engine._session_live_token_parts.extend(used_codes)\n\n                        reply_avatar_chunk_idx += 1", "                            if fm_engine.dump_motion:\n                                used_codes = [\n                                    s[\"reply_codes\"].detach().to(device=\"cpu\", dtype=torch.int16).contiguous()\n                                    for s in used_steps\n                                    if isinstance(s.get(\"reply_codes\"), torch.Tensor)\n                                ]\n                                if used_codes:\n                                    fm_engine._session_live_token_parts.extend(used_codes)\n\n                        reply_avatar_chunk_idx += 1"], ["                    except Exception as e:\n                        print(f\"[GPU] WARNING audio_q.put failed: {e!r}\", flush=True)\n\n                if prebuffer_chunks <= 0 and not prebuffer_ready.is_set():", "                    except Exception as e:\n                        print(f\"[GPU] WARNING audio_q.put failed: {e!r}\", flush=True)\n\n                # EMOTION_LIVE_PATCH_V4: batched queue enqueue.\n                def _enqueue_frame_batch(packets: list) -> None:\n                    \"\"\"Enqueue a whole chunk of frames in ONE event-loop round-trip.\n\n                    The per-packet first-per-turn marks are preserved, but the 24\n                    individual run_coroutine_threadsafe calls (each a cross-thread\n                    round-trip that blocks the GPU producer thread) collapse into a\n                    single one. Backpressure semantics are unchanged: the batched\n                    coroutine still awaits each frame_q.put, so a full queue blocks\n                    the producer exactly as before.\n                    \"\"\"\n                    if not packets:\n                        return\n                    for pkt in packets:\n                        reply_engine._mark_first_per_turn(\n                            \"server_avatar_frame_ready\", pkt.get(\"turn_id\"), at=pkt.get(\"t_ready\"),\n                        )\n\n                    async def _put_all() -> None:\n                        for pkt in packets:\n                            await frame_q.put(pkt)\n\n                    fut = asyncio.run_coroutine_threadsafe(_put_all(), event_loop)\n                    try:\n                        fut.result(timeout=FRAME_Q_PUT_TIMEOUT_S)\n                    except TimeoutError:\n                        print(\n                            f\"[GPU] WARNING frame_q batch put timeout ({len(packets)} frames)\",\n                            flush=True,\n                        )\n                    except Exception as e:\n                        print(f\"[GPU] WARNING frame_q batch put failed: {e!r}\", flush=True)\n\n                def _enqueue_audio_batch(packets: list) -> None:\n                    \"\"\"Audio twin of _enqueue_frame_batch (one round-trip per chunk).\"\"\"\n                    if not packets:\n                        return\n                    for pkt in packets:\n                        _pkt_turn = pkt.get(\"turn_id\")\n                        reply_engine._mark_first_per_turn(\n                            \"server_audio_queue_enter\", _pkt_turn,\n                            note=f\"audio_q depth at enqueue={audio_q.qsize()}\",\n                        )\n                        if _pkt_turn is not None:\n                            reply_engine.conv_logger.latency.queue_state(\n                                _pkt_turn, at_mark=\"server_audio_queue_enter\",\n                                audio_queue_depth=audio_q.qsize(),\n                                pending_audio_duration_s=round(audio_q.qsize() / float(args.fps), 3),\n                                input_backlog_s=round(reply_engine.input_backlog_sec(), 3),\n                            )\n\n                    async def _put_all() -> None:\n                        for pkt in packets:\n                            await audio_q.put(pkt)\n\n                    fut = asyncio.run_coroutine_threadsafe(_put_all(), event_loop)\n                    try:\n                        fut.result(timeout=FRAME_Q_PUT_TIMEOUT_S)\n                    except TimeoutError:\n                        print(\n                            f\"[GPU] WARNING audio_q batch put timeout ({len(packets)} packets)\",\n                            flush=True,\n                        )\n                    except Exception as e:\n                        print(f\"[GPU] WARNING audio_q batch put failed: {e!r}\", flush=True)\n\n                if prebuffer_chunks <= 0 and not prebuffer_ready.is_set():"], ["                        for frame_offset, audio_slice in enumerate(frame_audio[:n_frames]):\n                            _enqueue_audio({\n                                \"frame_number\": emitted + frame_offset,\n                                \"audio_pcm\": np.asarray(\n                                    audio_slice,\n                                    dtype=np.float32,\n                                ).copy(),\n                                \"turn_id\": _lat_turn_now,\n                            })", "                        _audio_batch = [\n                            {\n                                \"frame_number\": emitted + frame_offset,\n                                \"audio_pcm\": np.asarray(audio_slice, dtype=np.float32).copy(),\n                                \"turn_id\": _lat_turn_now,\n                            }\n                            for frame_offset, audio_slice in enumerate(frame_audio[:n_frames])\n                        ]\n                        _enqueue_audio_batch(_audio_batch)"], ["                            for pkt in packets:\n                                pkt[\"turn_id\"] = _lat_turn_now\n                                _enqueue_frame(pkt)", "                            for pkt in packets:\n                                pkt[\"turn_id\"] = _lat_turn_now\n                            _enqueue_frame_batch(packets)"], ["                            f\"abs={emitted}\"\n                        )\n\n                        if reply_avatar_chunk_idx % 50 == 0:", "                            f\"abs={emitted}\"\n                        )\n\n                        # Real-time budget watchdog: one chunk must be produced in\n                        # <= n_frames/fps seconds or the producer falls behind\n                        # permanently (it is pinned to real time by sender pacing).\n                        # Surface an explicit per-chunk warning with the cost\n                        # breakdown the moment the budget is exceeded, so the\n                        # bottleneck stage is identifiable directly from the log.\n                        _chunk_budget_ms = 1000.0 * float(n_frames) / float(args.fps)\n                        if produce_latency_ms > _chunk_budget_ms:\n                            hot_log.log(\n                                f\"[GPU] OVER-BUDGET chunk#{reply_avatar_chunk_idx}: \"\n                                f\"cost {produce_latency_ms:.0f}ms > budget {_chunk_budget_ms:.0f}ms \"\n                                f\"(moshi={moshi_total_ms:.0f} helium={float(fm_info['helium_ms']):.0f} \"\n                                f\"fm={float(fm_info['fm_ms']):.0f} render+jpeg={chunk_wall_ms:.0f}). \"\n                                f\"Producer cannot keep real time; mic backlog will grow.\"\n                            )\n\n                        if reply_avatar_chunk_idx % 50 == 0:"]]''')

START_WINNER_PATCHES = json.loads(r'''[["# STT + query routing + web search have been removed from this pipeline.\n", "# Emotion conditioning (emotion_conditioner + emotion LoRA)\n# EMOTION_ID: 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear (default: 0 = neutral)\nEMOTION_ID=\"${EMOTION_ID:-0}\"\nEMOTION_LORA_PATH=\"${EMOTION_LORA_PATH:-}\"\nEMOTION_LORA_RANK=\"${EMOTION_LORA_RANK:-32}\"\nEMOTION_LORA_ALPHA=\"${EMOTION_LORA_ALPHA:-64.0}\"\nEMOTION_CFG_SCALE=\"${EMOTION_CFG_SCALE:-3.0}\"\nEMOTION_CFG_BOOST_HAPPY=\"${EMOTION_CFG_BOOST_HAPPY:-1.0}\"\nif [[ -n \"$EMOTION_LORA_PATH\" && -f \"$EMOTION_LORA_PATH\" ]]; then\n  EMOTION_ARGS=(\n    --emotion_id \"$EMOTION_ID\"\n    --emotion_lora_path \"$EMOTION_LORA_PATH\"\n    --emotion_lora_rank \"$EMOTION_LORA_RANK\"\n    --emotion_lora_alpha \"$EMOTION_LORA_ALPHA\"\n    --emotion_cfg_scale \"$EMOTION_CFG_SCALE\"\n    --emotion_cfg_boost_happy \"$EMOTION_CFG_BOOST_HAPPY\"\n  )\nelse\n  # Still pass emotion_id even without a LoRA (uses base emotion_conditioner weights)\n  EMOTION_ARGS=(--emotion_id \"$EMOTION_ID\" --emotion_cfg_scale \"$EMOTION_CFG_SCALE\" --emotion_cfg_boost_happy \"$EMOTION_CFG_BOOST_HAPPY\")\nfi\n\n# AUTO_EMOTION (EMOTION_LIVE_PATCH_V3): detect the user's emotion from their\n# voice each utterance and drive the avatar's expression automatically.\n# EMOTION_ID above becomes the initial/fallback emotion until the first detection.\nAUTO_EMOTION=\"${AUTO_EMOTION:-0}\"\nif [[ \"$AUTO_EMOTION\" == \"1\" ]]; then\n  EMOTION_ARGS+=(\n    --auto_emotion\n    --ser_model \"${SER_MODEL:-emotion2vec/emotion2vec_plus_base}\"\n    --ser_device \"${SER_DEVICE:-cpu}\"\n    --ser_confidence_threshold \"${SER_CONFIDENCE_THRESHOLD:-0.5}\"\n    --ser_disgust_map \"${SER_DISGUST_MAP:-0}\"\n    --vad_rms_threshold \"${VAD_RMS_THRESHOLD:-0.02}\"\n    --vad_silence_sec \"${VAD_SILENCE_SEC:-0.7}\"\n    --vad_min_speech_sec \"${VAD_MIN_SPEECH_SEC:-0.5}\"\n    --vad_max_utterance_sec \"${VAD_MAX_UTTERANCE_SEC:-12.0}\"\n  )\nfi\n\n# SILENCE_EMOTION_PATCH_V1: speaking-state-aware CFG scales. While the avatar\n# is NOT speaking, the silence-seed audio guidance is raised to\n# A_CFG_SCALE_SILENCE (anchors the mouth closed) and the emotion guidance is\n# relaxed to EMOTION_CFG_SCALE_SILENCE (retains the expression without moving\n# the lips). Gate/emotion transitions ramp over the chunk counts below.\nA_CFG_SCALE_SILENCE=\"${A_CFG_SCALE_SILENCE:-3.0}\"\nEMOTION_CFG_SCALE_SILENCE=\"${EMOTION_CFG_SCALE_SILENCE:-1.5}\"\nGATE_CFG_RAMP_CHUNKS=\"${GATE_CFG_RAMP_CHUNKS:-2}\"\nEMOTION_SWITCH_RAMP_CHUNKS=\"${EMOTION_SWITCH_RAMP_CHUNKS:-2}\"\nEMOTION_ARGS+=(\n  --a_cfg_scale_silence \"$A_CFG_SCALE_SILENCE\"\n  --emotion_cfg_scale_silence \"$EMOTION_CFG_SCALE_SILENCE\"\n  --gate_cfg_ramp_chunks \"$GATE_CFG_RAMP_CHUNKS\"\n  --emotion_switch_ramp_chunks \"$EMOTION_SWITCH_RAMP_CHUNKS\"\n)\n\n# STT + query routing + web search have been removed from this pipeline.\n"], ["  \"${LORA_ARGS[@]}\" \\\n", "  \"${LORA_ARGS[@]}\" \\\n  \"${EMOTION_ARGS[@]}\" \\\n"], ["  LORA_ARGS=(\n    --lora_generator_path \"$LORA_GENERATOR_PATH\"\n    --lora_rank 64\n    --lora_alpha 128\n    --lora_dropout 0.05\n  )\nfi", "  LORA_ARGS=(\n    --lora_generator_path \"$LORA_GENERATOR_PATH\"\n    --lora_rank 64\n    --lora_alpha 128\n    --lora_dropout 0.05\n  )\nfi\n\n# --dump_motion records a rolling window of motion/audio for offline debugging.\n# It costs per-chunk GPU->CPU clones + PCIe transfers on the hot GPU thread plus\n# a ~1.4GB RAM window, so it is OFF by default; set DUMP_MOTION=1 to enable it.\nDUMP_MOTION=\"${DUMP_MOTION:-0}\"\nif [[ \"$DUMP_MOTION\" == \"1\" ]]; then\n  DUMP_MOTION_ARGS=(--dump_motion)\nelse\n  DUMP_MOTION_ARGS=()\nfi"], ["  --render_sub_batch 8 \\", "  --render_sub_batch \"${RENDER_SUB_BATCH:-8}\" \\"], ["  --dump_motion \\\n  --dump_dir \"$DUMP_DIR\" \\", "  \"${DUMP_MOTION_ARGS[@]}\" \\\n  --dump_dir \"$DUMP_DIR\" \\"]]''')

# ---------------------------------------------------------------------------
# Browser static files (AV_SMOOTH_PATCH_V1): bounded worklet emergency drop +
# frameStore purge. These ship pristine from the clone; patched in step 5b below.
# ---------------------------------------------------------------------------
WORKLET_PATCHES = json.loads(r'''[["    this.hardMaxBufferSamples = asSamples(1500);\n    this.emergencyTargetSamples = asSamples(500);\n    this.disableEmergencyDrop = true;", "    this.hardMaxBufferSamples = asSamples(1200);\n    this.emergencyTargetSamples = asSamples(500);\n    // AV_SMOOTH_PATCH_V1: bounded emergency drop. With the server pacing audio\n    // at exactly real time (1.0/fps) the queue can no longer ratchet upward, so\n    // normal operation sits far below this 1200ms cap and nothing is ever dropped.\n    // The drop only fires on a true overflow (a long stall flooding the queue),\n    // where skipping the oldest audio -- and letting the video, paced by\n    // audioStreamTime, jump with it -- keeps A/V in sync instead of drifting.\n    this.disableEmergencyDrop = false;"]]''')
INDEX_HTML_PATCHES = json.loads(r'''[["      const frameIdx = Math.max(0, frameNumber);\n      highestFrameReceived = Math.max(highestFrameReceived, frameIdx);\n      if (!frameStore.has(frameIdx)) {\n        const bitmap = await decodeFrameFromJpegBytes(jpegU8);\n        frameStore.set(frameIdx, { bitmap, chunkId, turnId, speaking });\n      }", "      const frameIdx = Math.max(0, frameNumber);\n      highestFrameReceived = Math.max(highestFrameReceived, frameIdx);\n      if (!frameStore.has(frameIdx)) {\n        const bitmap = await decodeFrameFromJpegBytes(jpegU8);\n        frameStore.set(frameIdx, { bitmap, chunkId, turnId, speaking });\n        // AV_SMOOTH_PATCH_V1: bound the store even when the render loop stalls.\n        // During an audio rebuffer audioStreamTime freezes, targetFrame stops\n        // advancing, and the render-side purge (which only runs when a frame is\n        // actually drawn) never fires, so ImageBitmaps (~1MB each) would\n        // accumulate without bound. Drop frames already behind the render cursor\n        // and hard-cap the store on every arrival.\n        for (const oldIdx of Array.from(frameStore.keys())) {\n          if (oldIdx < lastRenderedFrame - 1) frameStore.delete(oldIdx);\n        }\n        if (frameStore.size > 150) {\n          const keysAsc = Array.from(frameStore.keys()).sort((a, b) => a - b);\n          for (let i = 0; i < keysAsc.length - 150; i++) frameStore.delete(keysAsc[i]);\n        }\n      }"]]''')



def apply_patches(path, patch_list, marker, version_marker=None):
    """Apply (anchor, replacement) patches to `path`, idempotently.

    - If `marker` is already present, the file is considered patched and skipped.
    - If a *different* emotion patch version is present, the file is restored
      from its .bak_pre_emotion backup before re-patching.
    - Each anchor must occur exactly once, otherwise the cell fails loudly
      (a sign the upstream repo changed and the patches need refreshing).
    """
    text = path.read_text(encoding="utf-8")
    if marker in text:
        if version_marker is None or version_marker in text:
            print(f"[skip] {path.name} already patched")
            return False
        backup = path.with_suffix(path.suffix + ".bak_pre_emotion")
        if backup.exists():
            text = backup.read_text(encoding="utf-8")
            print(f"[restore] {path.name} restored from pre-emotion backup for re-patch")
    for i, (anchor, repl) in enumerate(patch_list, 1):
        n = text.count(anchor)
        if n != 1:
            raise RuntimeError(
                f"{path.name}: patch #{i} anchor found {n}x (expected exactly 1). "
                f"The upstream repo changed — refresh the patch anchors."
            )
        text = text.replace(anchor, repl)
    backup = path.with_suffix(path.suffix + ".bak_pre_emotion")
    if not backup.exists():
        backup.write_text(path.read_text(encoding="utf-8"), encoding="utf-8")
    path.write_text(text, encoding="utf-8")
    print(f"[patched] {path.name} ({len(patch_list)} patches)")
    return True


# 1. FM.py — EmotionConditioner + emotion_id plumbing (training-identical)
#    + streaming compositional-CFG plumbing (EMOTION_PATCH_V3)
#    + speaking-state-aware CFG rebalance (SILENCE_EMOTION_PATCH_V1)
apply_patches(GEN_DIR / "FM.py", FM_PATCHES, "EmotionConditioner", "SILENCE_EMOTION_PATCH_V1")

# 2. FMT.py — emotion bias kwargs on the transformer forward (training-identical)
#    + 3-way compositional CFG in forward_with_cfg (EMOTION_CFG_V1)
apply_patches(GEN_DIR / "FMT.py", FMT_PATCHES, "emotion_bias_now", "EMOTION_CFG_V1")

# 3. train_lora.py — append apply_emotion_lora_to_model (training-identical)
tl_path = GEN_DIR / "train_lora.py"
tl_text = tl_path.read_text(encoding="utf-8")
if "apply_emotion_lora_to_model" in tl_text:
    print("[skip] train_lora.py already patched")
else:
    tl_backup = tl_path.with_suffix(".py.bak_pre_emotion")
    if not tl_backup.exists():
        tl_backup.write_text(tl_text, encoding="utf-8")
    tl_path.write_text(tl_text.rstrip("\n") + "\n\n" + TRAIN_LORA_APPEND + "\n", encoding="utf-8")
    print("[patched] train_lora.py (apply_emotion_lora_to_model appended)")

# 4. Live server — --emotion_* args, emotion-aware _load_fm, fixed emotion_id
#    (EMOTION_LIVE_PATCH_V3 adds --auto_emotion, the mic tap and per-chunk emotion_id)
#    Emotion patches (SERVER_PATCHES[:11]) always apply. The AV-smoothness patches
#    (SERVER_PATCHES[11:], marker EMOTION_LIVE_PATCH_V4) are idempotent: the repo now
#    ships them, so they are skipped when the clone already carries them and only
#    re-apply if the base repo ever loses them.
apply_patches(SERVER_FILE, SERVER_PATCHES[:11], "EMOTION_LIVE_PATCH_V1", "SILENCE_EMOTION_PATCH_V1")
apply_patches(SERVER_FILE, SERVER_PATCHES[11:], "EMOTION_LIVE_PATCH_V4")

# 5. start_winner_live.sh — EMOTION_ARGS from env vars (+ AUTO_EMOTION args)
#    Same split: emotion args ([:2]) always apply; the DUMP_MOTION / render_sub_batch
#    smoothness patches ([2:], marker DUMP_MOTION) are idempotent.
apply_patches(START_WINNER_SH, START_WINNER_PATCHES[:2], "EMOTION_ARGS", "SILENCE_EMOTION_PATCH_V1")
apply_patches(START_WINNER_SH, START_WINNER_PATCHES[2:], "DUMP_MOTION")

# 5b. Browser static files -- bounded worklet emergency drop + frameStore purge
#     (AV_SMOOTH_PATCH_V1). Ships pristine from the clone; patched here.
AUDIO_WORKLET_JS = Path(IMTALKER_DIR) / "static" / "assets" / "audio-processor-aj-nodrop.js"
INDEX_HTML_FILE = Path(IMTALKER_DIR) / "static" / "index_v3_binary_fullscreen_aj_nodrop.html"
apply_patches(AUDIO_WORKLET_JS, WORKLET_PATCHES, "AV_SMOOTH_PATCH_V1")
apply_patches(INDEX_HTML_FILE, INDEX_HTML_PATCHES, "AV_SMOOTH_PATCH_V1")

# 6. Syntax-check the patched Python files with the venv interpreter
for py in [GEN_DIR / "FM.py", GEN_DIR / "FMT.py", GEN_DIR / "train_lora.py", SERVER_FILE,
           Path(IMTALKER_DIR) / "auto_emotion.py"]:
    compile(py.read_text(encoding="utf-8"), str(py), "exec")
print("[ok] all patched Python files compile")
print("[ok] emotion support injected — the repo now matches the adapter's training architecture")


## Step 13 — Validate the emotion adapter + the patched stack

Two checks before launch:

1. **Patch integrity** — confirms `FM.py` (`EmotionConditioner` + `EMOTION_PATCH_V3`
   emotion_cfg_scale plumbing), `FMT.py` (`emotion_bias_now` + `EMOTION_CFG_V1`
   compositional CFG), `train_lora.py` (`apply_emotion_lora_to_model`), the
   live server (`--emotion_*` args incl. `--emotion_cfg_scale` + fixed `emotion_id`
   in the sample dict) and `start_winner_live.sh` (`EMOTION_ARGS` incl.
   `EMOTION_CFG_SCALE`) all carry the injected code.
2. **Adapter integrity** — loads `best.ckpt` with the venv torch and verifies
   it contains EMA weights, `emotion_conditioner.*` tensors and LoRA A/B
   tensors, then prints the per-emotion conditioning-bias norms and pairwise
   emotion-vector distances (the same check used after training — distinct
   vectors mean the adapter actually learned to separate the emotions).


In [ ]:
# --- 1. Patch integrity -------------------------------------------------------
PATCH_CHECKS = [
    (GEN_DIR / "FM.py", ["EmotionConditioner", "EMOTION_PATCH_V3", "emotion_conditioner",
                         "emotion_cfg_scale=_emotion_cfg_scale", "emotion_cfg_boost_happy",
                         "SILENCE_EMOTION_PATCH_V1", "speaking_weight", "emotion_blend"]),
    (GEN_DIR / "FMT.py", ["emotion_bias_now", "emotion_bias_prev",
                          "EMOTION_CFG_V1", "emotion_cfg_scale=None", "chunks=3"]),
    (GEN_DIR / "train_lora.py", ["apply_emotion_lora_to_model"]),
    (SERVER_FILE, ["EMOTION_LIVE_PATCH_V1", "EMOTION_LIVE_PATCH_V2", "EMOTION_LIVE_PATCH_V3",
                   "--emotion_id", "--emotion_lora_path", "--emotion_cfg_scale", "--emotion_cfg_boost_happy",
                   "--auto_emotion", "get_auto_emotion_manager", "init_auto_emotion_manager",
                   "apply_emotion_lora_to_model", "emotion lora loaded",
                   "_auto_emotion_mgr.feed(_pcm_i16, int(input_sr))",
                   "EMOTION_LIVE_PATCH_V4", "_enqueue_frame_batch", "_enqueue_audio_batch",
                   "OVER-BUDGET chunk#", "min_send_interval_s = max(0.001, 1.0 / float(args.fps))",
                   "SILENCE_EMOTION_PATCH_V1", "--a_cfg_scale_silence", "speaking=bool(assistant_active)"]),
    (START_WINNER_SH, ["EMOTION_ARGS", '--emotion_id "$EMOTION_ID"', '"${EMOTION_ARGS[@]}"',
                       "EMOTION_CFG_SCALE", "EMOTION_CFG_BOOST_HAPPY", '--emotion_cfg_scale "$EMOTION_CFG_SCALE"',
                       "AUTO_EMOTION", "--auto_emotion",
                       "DUMP_MOTION_ARGS", '"${DUMP_MOTION_ARGS[@]}"', "RENDER_SUB_BATCH:-8", '--a_cfg_scale_silence "$A_CFG_SCALE_SILENCE"']),
    (Path(IMTALKER_DIR) / "auto_emotion.py",
     ["class AutoEmotionManager", "class EnergyVAD", "class SpeechEmotionClassifier",
      "class EmotionLogWriter", "get_auto_emotion_manager", "init_auto_emotion_manager"]),
    (Path(IMTALKER_DIR) / "static" / "assets" / "audio-processor-aj-nodrop.js",
     ["AV_SMOOTH_PATCH_V1", "disableEmergencyDrop = false", "asSamples(1200)"]),
    (Path(IMTALKER_DIR) / "static" / "index_v3_binary_fullscreen_aj_nodrop.html",
     ["AV_SMOOTH_PATCH_V1", "frameStore.size > 150"]),
]
patch_ok = True
for path, needles in PATCH_CHECKS:
    text = path.read_text(encoding="utf-8")
    missing = [n for n in needles if n not in text]
    status = "OK" if not missing else f"MISSING {missing}"
    patch_ok = patch_ok and not missing
    print(f"  [{'OK' if not missing else 'FAIL'}] {path.name}: {status if missing else 'all hooks present'}")
if not patch_ok:
    raise RuntimeError("Emotion patches incomplete — re-run Step 12 (injection)")
print("[ok] all emotion patches verified in the cloned repo")

# --- 2. Adapter integrity (runs in the venv, same check as post-training) -----
EVAL_SRC = r'''
import sys
from pathlib import Path
import torch
import torch.nn.functional as F

ckpt_path = Path(sys.argv[1])
ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
state = ckpt.get("ema_state_dict") or ckpt.get("state_dict", ckpt)
if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
    state = state["model"]
state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in state.items()}

emo_keys = [k for k in state if k.startswith("emotion_conditioner.")]
lora_a = [k for k in state if ".lora_A.weight" in k]
lora_b = [k for k in state if ".lora_B.weight" in k]
print(f"checkpoint keys : {len(state)}")
print(f"  EMA weights   : {'yes' if 'ema_state_dict' in ckpt else 'no (raw state_dict)'}")
print(f"  emotion_conditioner tensors: {len(emo_keys)}")
print(f"  LoRA A / B tensors         : {len(lora_a)} / {len(lora_b)}")
assert emo_keys, "no emotion_conditioner weights in checkpoint"
assert lora_a and lora_b, "no LoRA weights in checkpoint"

W = state["emotion_conditioner.emotion_projection.0.weight"].float()
b = state["emotion_conditioner.emotion_projection.0.bias"].float()
ln_w = state["emotion_conditioner.emotion_projection.1.weight"].float()
ln_b = state["emotion_conditioner.emotion_projection.1.bias"].float()

NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
biases = {}
print("\nPer-emotion conditioning bias norms:")
for eid, name in NAMES.items():
    oh = F.one_hot(torch.tensor(eid), num_classes=7).float()
    h = F.silu(F.layer_norm(W @ oh + b, (W.shape[0],), ln_w, ln_b))
    biases[name] = h
    print(f"  {name:10s}: {h.norm().item():.4f}")

print("\nPairwise emotion-vector distances (grow as emotions separate):")
names = list(NAMES.values())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        dist = (biases[names[i]] - biases[names[j]]).norm().item()
        print(f"  {names[i]:10s} vs {names[j]:10s}: {dist:.4f}")

dists = [(biases[a] - biases[b]).norm().item() for a in names for b in names if a != b]
if max(dists) < 1e-3:
    print("\n[warn] emotion vectors are nearly identical - the adapter has not learned to separate emotions.")
else:
    print(f"\n[ok] emotion vectors are distinct (max distance {max(dists):.4f}).")
'''

eval_path = Path("/tmp/eval_emotion_ckpt.py")
eval_path.write_text(EVAL_SRC, encoding="utf-8")
proc = subprocess.run([VENV_PYTHON, str(eval_path), str(EMOTION_ADAPTER_PATH)],
                      capture_output=True, text=True)
print(proc.stdout)
assert proc.returncode == 0, proc.stderr[-2000:]

print(f"[ok] emotion adapter validated: {EMOTION_ADAPTER_PATH}")
print(f"[ok] active emotion for this session: {EMOTION} (id={EMOTION_ID})")

# --- 3. SER smoke test (auto emotion) ---------------------------------------
if AUTO_EMOTION:
    SER_SMOKE_SRC = r'''
import sys
import numpy as np

sys.path.insert(0, sys.argv[1])
from auto_emotion import SpeechEmotionClassifier

clf = SpeechEmotionClassifier(model_name=sys.argv[2], device="cpu", half=False, disgust_map=0)
rng = np.random.default_rng(0)
# 1s of synthetic speech-like noise: verifies model load + forward + label map
pcm = (0.05 * rng.standard_normal(24000)).astype(np.float32)
emotion_id, label, conf = clf.predict(pcm)
print(f"smoke: label={label} emotion_id={emotion_id} conf={conf:.3f}")
assert 0 <= int(emotion_id) <= 5, f"invalid emotion_id {emotion_id}"
print("[ok] SER smoke test passed")
'''
    smoke_path = Path("/tmp/ser_smoke_test.py")
    smoke_path.write_text(SER_SMOKE_SRC, encoding="utf-8")
    proc = subprocess.run(
        [VENV_PYTHON, str(smoke_path), str(IMTALKER_DIR), SER_MODEL],
        capture_output=True, text=True, timeout=600,
    )
    print(proc.stdout)
    assert proc.returncode == 0, proc.stderr[-2000:]
    print(f"[ok] auto-emotion SER model validated: {SER_MODEL}")
else:
    print("[skip] AUTO_EMOTION is False - SER smoke test skipped")


## Step 14 — Pre-launch validation

GPU, CUDA, the full asset table (now including the emotion adapter), and port
availability (with automatic recovery if a *previous run of this same notebook*
is still holding the port).


In [ ]:
def check_port_and_recover(port):
    if not port_listening("127.0.0.1", port):
        print(f"[ok] port {port} is free")
        return
    pids_on_port = find_pids_on_port(port)
    prior_pid = None
    if os.path.exists(PID_PATH):
        prior_pid = open(PID_PATH).read().strip()
    if prior_pid and prior_pid in pids_on_port:
        print(f"[recovery] port {port} held by a previous notebook-launched server (pid {prior_pid}); terminating it")
        subprocess.run(["kill", "-9", prior_pid])
        time.sleep(2)
        if port_listening("127.0.0.1", port):
            raise RuntimeError(f"Port {port} still in use after terminating prior pid {prior_pid}")
        print(f"[ok] port {port} freed")
    else:
        raise RuntimeError(
            f"Port {port} is already in use by pid(s) {pids_on_port}, which were not started by this "
            f"notebook. Stop that process manually (or choose a different PORT) before launching."
        )


check_port_and_recover(PORT)

torch_info = assert_torch_pin()
if not torch_info["cuda_available"]:
    raise RuntimeError("CUDA not available inside venv torch at launch time")
if torch_info["device_name"] and "5090" in torch_info["device_name"]:
    print(f"[ok] GPU confirmed: {torch_info['device_name']}")
else:
    print(f"[warn] GPU device name '{torch_info['device_name']}' does not mention 5090 — continuing anyway")

missing = [a["path"] for a in REQUIRED_ASSETS if not os.path.exists(a["path"])]
if not os.path.exists(VOICE_PROMPT_PATH):
    missing.append(VOICE_PROMPT_PATH)
if not os.path.exists(EMOTION_ADAPTER_PATH):
    missing.append(EMOTION_ADAPTER_PATH)
if missing:
    raise RuntimeError(f"Missing required assets immediately before launch: {missing}")

print("[ok] pre-launch validation passed")


## Step 15 — Inspect `run_live.sh` and resolve runtime overrides

The notebook never reimplements `run_live.sh`'s argument list — it reads and
prints the actual script, then prepares only the environment-variable
overrides the script (and the `start_winner_live.sh` it delegates to) already
support: `REF_PATH`, `A_CFG_SCALE`, `NFE`, `PROMPT_FILE`, `VOICE_PROMPT`,
`VOICE_PROMPT_DIR`, `LORA_GENERATOR_PATH`, `DISABLE_LORA`, plus
`HOST`/`PORT`/`VENV_DIR` — and, after the Step 12 patch, the emotion overrides
`EMOTION_ID` / `EMOTION_LORA_PATH` / `EMOTION_LORA_RANK` / `EMOTION_LORA_ALPHA` /
`EMOTION_CFG_SCALE`.

> Leave `A_CFG_SCALE` unset (AH default ~1.13) — it drives lip-sync. The facial
> expression strength is controlled independently by `EMOTION_CFG_SCALE`
> (default 3.0), which the Step 12 compositional-CFG patch applies to the emotion
> delta only. Do **not** try to fix a neutral face by raising `A_CFG_SCALE` — that
> would over-amplify the audio conditioning and degrade lip-sync.

> The new `run_live.sh` reads `VENV_DIR` (not the old `VENV`), and it no
> longer accepts a raw `TEXT_PROMPT` string — the system prompt is always
> read from a file (`PROMPT_FILE`, defaulting to
> `IMTalker/prompts/RB_Robert_System_Prompt_full.txt`).


In [ ]:
with open(os.path.join(PROJECT_ROOT, "run_live.sh")) as f:
    run_live_sh_contents = f.read()
print(run_live_sh_contents)


In [ ]:
env_overrides = {
    "HOST": HOST,
    "PORT": str(PORT),
    "VENV_DIR": VENV_DIR,
    "SPEECH2AVATAR_ROOT": PROJECT_ROOT,
    # Emotion conditioning — read by the patched start_winner_live.sh, which
    # turns them into --emotion_* server args (see Step 12).
    "EMOTION_ID": str(EMOTION_ID),
    "EMOTION_LORA_PATH": EMOTION_ADAPTER_PATH,
    "EMOTION_LORA_RANK": str(EMOTION_LORA_RANK),
    "EMOTION_LORA_ALPHA": str(EMOTION_LORA_ALPHA),
    # Emotion-only CFG scale (compositional guidance; audio CFG stays at A_CFG_SCALE).
    "EMOTION_CFG_SCALE": str(EMOTION_CFG_SCALE),
    "EMOTION_CFG_BOOST_HAPPY": str(EMOTION_CFG_BOOST_HAPPY),
    # Silence-aware emotion CFG (SILENCE_EMOTION_PATCH_V1): rebalanced CFG
    # scales while the avatar is not speaking (see Parameters cell).
    "A_CFG_SCALE_SILENCE": str(A_CFG_SCALE_SILENCE),
    "EMOTION_CFG_SCALE_SILENCE": str(EMOTION_CFG_SCALE_SILENCE),
    "GATE_CFG_RAMP_CHUNKS": str(GATE_CFG_RAMP_CHUNKS),
    "EMOTION_SWITCH_RAMP_CHUNKS": str(EMOTION_SWITCH_RAMP_CHUNKS),
    # AUTO_EMOTION (V3) - voice-based automatic emotion detection.
    "AUTO_EMOTION": "1" if AUTO_EMOTION else "0",
    "SER_MODEL": SER_MODEL,
    "SER_DEVICE": SER_DEVICE,
    "SER_CONFIDENCE_THRESHOLD": str(SER_CONFIDENCE_THRESHOLD),
    "VAD_RMS_THRESHOLD": str(VAD_RMS_THRESHOLD),
    "VAD_SILENCE_SEC": str(VAD_SILENCE_SEC),
    "VAD_MIN_SPEECH_SEC": str(VAD_MIN_SPEECH_SEC),
    "VAD_MAX_UTTERANCE_SEC": str(VAD_MAX_UTTERANCE_SEC),
    # Renderer sub-batch: frames decoded per render call. The renderer's attention
    # needs ~0.5 GiB/frame; on a loaded 32 GB 5090 only ~4.5 GiB is free after the
    # models, so a whole 24-frame chunk (12 GiB) OOMs. 8 is the safe maximum.
    "RENDER_SUB_BATCH": "8",
}
if REF_PATH:
    env_overrides["REF_PATH"] = REF_PATH
if A_CFG_SCALE:
    env_overrides["A_CFG_SCALE"] = str(A_CFG_SCALE)
if NFE:
    env_overrides["NFE"] = str(NFE)
if PROMPT_FILE:
    env_overrides["PROMPT_FILE"] = PROMPT_FILE
if VOICE_PROMPT:
    env_overrides["VOICE_PROMPT"] = VOICE_PROMPT
if VOICE_PROMPT_DIR:
    env_overrides["VOICE_PROMPT_DIR"] = VOICE_PROMPT_DIR
if LORA_GENERATOR_PATH:
    env_overrides["LORA_GENERATOR_PATH"] = LORA_GENERATOR_PATH
if DISABLE_LORA:
    env_overrides["DISABLE_LORA"] = str(DISABLE_LORA)
if JITTER_BUFFER_SEC:
    env_overrides["JITTER_BUFFER_SEC"] = str(JITTER_BUFFER_SEC)


print("Resolved environment overrides for run_live.sh:")
for k, v in env_overrides.items():
    print(f"  {k}={v}")


## Step 16 — Launch the live server

Runs `bash run_live.sh` in the background (it sources the venv itself), logs
to `live_server.log`, and records the PID for the stop/recovery cells below.
The selected emotion (`EMOTION_ID`) and the emotion adapter path are passed
through the environment into the patched `start_winner_live.sh`.


In [ ]:
launch_env = os.environ.copy()
launch_env.update(env_overrides)

log_file = open(LOG_PATH, "w")
live_proc = subprocess.Popen(
    ["bash", "run_live.sh"], cwd=PROJECT_ROOT, env=launch_env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
with open(PID_PATH, "w") as f:
    f.write(str(live_proc.pid))

print(f"[ok] launched live server: pid={live_proc.pid}")
print(f"     log file: {LOG_PATH}")
print(f"     pid file: {PID_PATH}")


## Step 17 — Wait for healthy startup

Polls the log for the documented healthy markers (now including the
`emotion lora loaded` line printed by the patched `_load_fm`) and the listening
port simultaneously. On timeout or early process exit, it dumps the log tail
and common-failure hints (CUDA OOM, missing path, traceback) instead of
hanging forever.


In [ ]:
HEALTHY_MARKERS = {
    "emotion lora loaded": "emotion lora loaded",
    "using direct Moshi reply hidden": "using direct Moshi reply hidden",
    "installed PersonaPlex graphed hidden capture": "installed PersonaPlex graphed hidden capture",
    "Uvicorn running on host:port": f"Uvicorn running on http://{HOST}:{PORT}",
    "serving static html": "index_v3_binary_fullscreen_aj_nodrop.html",
}
if not DISABLE_LORA:
    # Only printed when the style LoRA is merged into the base weights first
    # (the default both-LoRAs branch of the patched _load_fm).
    HEALTHY_MARKERS["style LoRA merged into base"] = "style LoRA layers"

ERROR_HINTS = {
    "CUDA out of memory": "GPU out of memory — reduce concurrent load or confirm the pod truly has an RTX 5090 with enough VRAM",
    "Traceback (most recent call last)": "a Python exception occurred during startup — see the traceback above in the log",
    "No such file or directory": "a referenced checkpoint/asset path is wrong — re-run Step 10's asset table",
    "CUDA error": "a CUDA/driver mismatch occurred — re-check Step 2 (nvidia-smi) and the torch CUDA build",
    "Address already in use": "the port was taken by another process after the Step 14 check — re-run Step 14",
    "Cannot find": "the VOICE_PROMPT file was not found under VOICE_PROMPT_DIR — re-run Step 10 to fetch/extract voices.tgz",
    "Missing required path": "start_winner_live.sh could not find a required checkpoint/prompt/HTML path — re-run Step 10 and check its printed list",
    "emotion_lora_path not found": "the emotion adapter file is missing — re-run Step 10 (asset download) and Step 13 (validation)",
    "cannot import name 'apply_emotion_lora_to_model'": "train_lora.py was not patched — re-run Step 12 (emotion injection)",
    "anchor found": "an emotion patch anchor no longer matches the repo — the upstream repo changed; refresh the Step 12 patches",
    "[AUTO-EMOTION] DISABLED": "the SER model failed to load - the server is running with the fixed EMOTION fallback; check the traceback above (HF download, VRAM, funasr/transformers import)",
}


def wait_for_healthy(timeout=STARTUP_TIMEOUT_SEC, poll_interval=POLL_INTERVAL_SEC):
    start = time.time()
    last_print = 0.0
    while time.time() - start < timeout:
        if live_proc.poll() is not None:
            tail_log(120)
            raise RuntimeError(
                f"live server process exited early with code {live_proc.returncode}; see log tail above"
            )

        log_text = ""
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, "r", errors="ignore") as f:
                log_text = f.read()

        markers_ok = {label: (needle in log_text) for label, needle in HEALTHY_MARKERS.items()}
        port_ok = port_listening("127.0.0.1", PORT)

        if all(markers_ok.values()) and port_ok:
            print("[ok] live server is healthy")
            for label, ok in markers_ok.items():
                print(f"  [OK] {label}")
            print(f"  [OK] port {PORT} listening")
            return True

        if time.time() - last_print > 15:
            elapsed = int(time.time() - start)
            print(f"[..] waiting ({elapsed}s/{timeout}s) markers={markers_ok} port_listening={port_ok}")
            last_print = time.time()

        time.sleep(poll_interval)

    print("[error] timed out waiting for healthy startup; log tail:")
    tail_log(150)
    log_text = ""
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, "r", errors="ignore") as f:
            log_text = f.read()
    for needle, hint in ERROR_HINTS.items():
        if needle in log_text:
            print(f"[diagnosis] found '{needle}' in log -> {hint}")
    raise TimeoutError("Live server did not become healthy within the timeout; see diagnostics above")


wait_for_healthy()

# Auto-emotion sanity: warn (don't fail) if the SER model didn't come up.
if AUTO_EMOTION and os.path.exists(LOG_PATH):
    with open(LOG_PATH, "r", errors="ignore") as f:
        _log = f.read()
    if "[AUTO-EMOTION] DISABLED" in _log:
        print("[warn] AUTO_EMOTION is on but the SER model failed to load - the avatar will use")
        print("       the fixed EMOTION fallback. See '[AUTO-EMOTION] DISABLED' in the log above.")
    elif "[AUTO-EMOTION] enabled:" in _log:
        print("[ok] auto-emotion SER model is active (voice-driven expressions)")


## Step 18 — HTTP / Uvicorn confirmation

Final confirmation that the FastAPI/Uvicorn app is actually serving HTTP on
`0.0.0.0:{PORT}` (default `8998`), not just that the port is open. This only
checks `127.0.0.1` inside the pod — it does **not** prove the RunPod public
proxy URL works, since that depends on `PORT` being one of the ports actually
exposed by your pod template (see the note in the Parameters cell). This cell
prints the expected public URL from `RUNPOD_POD_ID` + `PORT` for convenience,
plus the emotion that is now baked into the running server.


In [ ]:
import urllib.error
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10) as resp:
        body = resp.read(2000).decode(errors="ignore")
        print(f"[ok] HTTP {resp.status} from http://127.0.0.1:{PORT}/")
        print(body[:500])
except urllib.error.URLError as e:
    tail_log(80)
    raise RuntimeError(f"HTTP check failed against http://127.0.0.1:{PORT}/: {e}")

with open(LOG_PATH, "r", errors="ignore") as f:
    final_log = f.read()
assert f"Uvicorn running on http://{HOST}:{PORT}" in final_log, "Uvicorn startup banner not found in log"
assert "emotion lora loaded" in final_log, "emotion LoRA load marker not found in log"

print()
print("=" * 72)
print("SUCCESS: PersonaPlex + IMTalker emotion live server (AH AudioPace) is running and healthy")
print(f"  Emotion: {EMOTION} (id={EMOTION_ID}) — fixed for this entire session")
print(f"  Emotion adapter: {EMOTION_ADAPTER_PATH}")
print(f"  Internal: http://{HOST}:{PORT}")
if PUBLIC_URL:
    print(f"  Public:   {PUBLIC_URL}")
    print(f"  (auto-built from RUNPOD_POD_ID={RUNPOD_POD_ID} and PORT={PORT} — confirm this exact")
    print(f"   port is listed under this pod's Connect -> HTTP Service on the RunPod dashboard;")
    print(f"   the proxy only forwards ports that were exposed when the pod was created/edited)")
else:
    print(f"  Open port {PORT} through your RunPod pod's proxy/port mapping to access the browser UI")
    print(f"  (RUNPOD_POD_ID was not auto-detected; set it in the Parameters cell for a printed URL)")
print(f"  Browser UI file: static/index_v3_binary_fullscreen_aj_nodrop.html")
print(f"  PID: {open(PID_PATH).read().strip()}   Log: {LOG_PATH}")
print("=" * 72)


## Operational cells (run any time)

These are safe to re-run independently after the server is up.


In [ ]:
# Tail the live server log
tail_log(200)


In [ ]:
# Full diagnostics — safe to run any time, including during a failure
def run_full_diagnostics():
    print("== GPU ==")
    run("nvidia-smi", check=False)

    print("\n== Torch / CUDA (venv) ==")
    try:
        print(get_torch_info())
    except Exception as e:
        print(f"[error] torch check failed: {e}")

    print(f"\n== Port {PORT} ==")
    print(f"listening: {port_listening('127.0.0.1', PORT)}  pids: {find_pids_on_port(PORT)}")

    print("\n== Required assets ==")
    for asset in REQUIRED_ASSETS:
        print(f"  [{'OK' if os.path.exists(asset['path']) else 'MISSING'}] {asset['path']}")
    print(f"  [{'OK' if os.path.exists(VOICE_PROMPT_PATH) else 'MISSING'}] {VOICE_PROMPT_PATH}")
    print(f"  [{'OK' if os.path.exists(EMOTION_ADAPTER_PATH) else 'MISSING'}] {EMOTION_ADAPTER_PATH}")

    print("\n== Emotion patch status ==")
    for path, needles in PATCH_CHECKS:
        text = path.read_text(encoding="utf-8") if path.exists() else ""
        missing = [n for n in needles if n not in text]
        print(f"  [{'OK' if not missing else 'FAIL'}] {path.name}" + (f" missing: {missing}" if missing else ""))

    print("\n== pip check (venv) ==")
    run(f"{VENV_ACTIVATE} && pip check", check=False)

    print("\n== Log tail ==")
    tail_log(100)


run_full_diagnostics()


In [ ]:
# Stop switch — set STOP_SERVER = True and re-run this cell to terminate the live server.
STOP_SERVER = False


def stop_server():
    if not os.path.exists(PID_PATH):
        print("[info] no pid file found; nothing to stop")
        return
    pid = open(PID_PATH).read().strip()
    print(f"[stop] terminating live server pid {pid}")
    subprocess.run(["kill", "-9", pid])
    os.remove(PID_PATH)
    print("[ok] server stopped")


if STOP_SERVER:
    stop_server()
else:
    print("STOP_SERVER is False — set it to True above and re-run this cell to stop the server.")


In [ ]:
# How emotion control works with AUTO_EMOTION
# --------------------------------------------
# AUTO_EMOTION = True (default): the avatar's expression is driven by the USER'S
# VOICE. Each user utterance is classified by the SER model and the face
# switches automatically (~1s after the utterance ends). EMOTION in the
# Parameters cell is only the initial/fallback emotion - no restart needed,
# no manual selection, works for the whole conversation.
#
# AUTO_EMOTION = False: the old manual behaviour - EMOTION is baked in at
# launch. To change it: run the stop cell, edit EMOTION in Parameters, re-run
# Parameters, then re-run Launch (Step 16).
NEW_EMOTION = "sad"  # manual mode only: "neutral", "happy", "sad", "angry", "surprise", "fear"

print(f"AUTO_EMOTION = {AUTO_EMOTION}")
if AUTO_EMOTION:
    print("The avatar mirrors the user's detected voice emotion automatically.")
    print(f"Initial/fallback emotion: {EMOTION} (id={EMOTION_ID})")
    print(f"SER model: {SER_MODEL} | confidence threshold: {SER_CONFIDENCE_THRESHOLD}")
    print("Watch the server log for lines like:")
    print("  [AUTO-EMOTION] utterance 2.31s -> happy (p=0.87) [neutral -> happy] switching")
    print()
    print("Prediction log (one CSV row per user utterance):")
    print("  IMTalker/emotion_logs/emotion_log_<timestamp>.csv")
    print("  columns: timestamp, session, utterance, duration_sec, detected_label,")
    print("           detected_emotion, confidence, accepted, previous_emotion,")
    print("           applied_emotion  <- the emotion sent to the avatar model")
else:
    print(f"Currently running: EMOTION={EMOTION} (id={EMOTION_ID}) - fixed for the session")
    print()
    print(f"To switch to '{NEW_EMOTION}':")
    print(f"  1. Run the STOP SERVER cell above and wait for the port to free")
    print(f"  2. In the Parameters cell, set:  EMOTION = \"{NEW_EMOTION}\"")
    print(f"  3. Re-run the Parameters cell, then the Launch cell (Step 16)")
print()
print("Emotion reference:")
for _name, _id in EMOTION_MAP.items():
    marker = "  <-- initial" if _id == EMOTION_ID else ""
    print(f"  {_id}: {_name}{marker}")
print()
print(f"Expression strength: EMOTION_CFG_SCALE={EMOTION_CFG_SCALE} (edit in Parameters;")
print("  raise for a stronger expression, lower toward 1.0 for subtler, then re-launch)")
